# GNR Step 3 v5.1 — Preserve Photoshop; local A/M/B staircase edits
Updated from the supplied v4 notebook using the latest staircase diagram.

**Regression correction:** the earlier broad gap/pocket-filling stages could merge shapes and erase white details. They are OFF by default. Cell 10 always starts from the original Photoshop draft and ignores intermediate RED_FINAL/RED_SEP variables. Only checked, bounded corner edits are accepted; enclosed white regions and white far from red are protected. Cumulative automatic changes are limited to 1% by default. This is an edit budget, not a measure of remaining designer work.

**Reference:** flat A → horizontal 3,2,2 → shared M (1 pixel) → vertical 2,2,3 → flat B. From either flat toward M, run lengths start at 2/3/4, hold or decrease one level to 1. Flat lengths are free; there is no step-count dominance. This version requires a single shared transition for the illustrated corner family. Ambiguous junctions and other corner families remain unresolved.

**Start here:** run V5 SETUP and V5 SELF-TEST, then original Cells 1–3 and Cell 10. Cells 4–6 are optional diagnostic previews. CPU is sufficient. Set DESIGN in Cell 2. In Cell 3, reuse the inbox file, paste an existing draft's full Drive path into AI_DRAFT_PATH, or upload one when prompted. The finisher requires that draft; it does not generate Step_3 from Step_2 in its default mode. Use Cell 2's Photoshop prompt first if you do not yet have a draft. No retraining or API key is required with the defaults.

**Optional:** enable V5_USE_LEARNED_FINISHER to propose with an existing saved finisher through Cells 9–10. Its proposal is rejected if it violates protection, edit budget or affected-contour checks. Enable V5_RUN_LLM_REVIEW for paid Cell 11; its changes have the same original-image budget/protection. Broad fill previews require explicit V5_ENABLE_GAP_REDRAW/V5_ENABLE_POCKET_FILL flags and are never selected automatically as the final baseline. Training/model experiments and unknown external scripts remain disabled.

The embedded constructor searches bounded native-pixel paths, checks the rendered contour and preserves already-passing corners. Re-extraction follows each accepted edit. Cleanup and LLM replacements must pass checks on complete affected raster regions. Anchors are inferred from orthogonal flat extrema, not from the centre of a crop. This inference is conservative and does not establish semantic identity.

Exports go to **02_Review_Output_v5**, with PNG/BMP and a JSON report of passed, failed and unresolved detected corners and narrow raster-region gaps. Native dimensions are checked against the original Step_2 file. All exports remain review drafts: semantic applicability, intentional white and the completeness of curve detection require designer review. The original notebook and its legacy shared library are preserved.

Old saved outputs were cleared so they cannot be mistaken for measurements from this update. Source-update tests do not establish full-image quality or the 10% manual-work target.


In [ ]:
# V5 SETUP — only installs missing geometry dependencies; no Drive/API/training calls.
import base64,zlib,sys,subprocess,importlib.util,types
from pathlib import Path
needed={"numpy":"numpy", "scipy":"scipy", "PIL":"Pillow", "cv2":"opencv-python-headless", "skimage":"scikit-image", "matplotlib":"matplotlib"}
missing=[package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,"-m","pip","install",*missing])
V5_CORE_SOURCE=zlib.decompress(base64.b64decode('eJy9fe1y28aW4H89BexUDMACKZGy72RJw7WKrTjasWWV7GTuLosXBZJNEhEIIABIifHYNb/2AfYh9sHmSfZ8dDcaAEkp985dVWIS6K/Tp893n24+ffr0/OTDyY9WEpbRRnSy6F7E1jTNE5FbizxcrcLcsybpOpmJGbxPijJfT8s0t8JkZuViE4k7axGWougeHX1eRoU1T+M4vSuscimsdSFyu7BiLC+tWRRijwOrWOZRcmvl68R6f3H17vPPn6x5nq4sEUGj3JpD/aMwmS5hlDK9C/OZFUKbMAcI0kTBWOZhUkRllCZd6zM0ExYMnqRWUYqsMwWAYcB0FSXQkege/QL/5mUYJVYhFiuRlCG2xCa5yNK8FDMPGpdWFhaFmHWtT2IVJmU0tcIsi6NpOIniqNxC51EcH+Xi93WUi8KaiSJaIKYYEV3rKrVW6UzE1JmYpOmtNRVxbMFMzq8vcThxL6ZrGM5CkLtHT58+PaK5TwFtYopAFVa0QpCsNzgJAeifiXm4jstZNC258nydwBqksa4a5+tgGk6X4ki+WIbFMo4m6jFK1bffijThTrKwxCqqi2t4VJVycaS+Tjd99TVZr7KtFQKWM+6hmEbwQpXOolW4EFQ+i7jC9eV7VXyJhUdHv17cfLr8eGX5lv2y2+uedpCkRL5h6gtXE/vo6gcoTbIuLHXhOGfemesBImP36OoFF4R5Hm6d0ejU63mnY2/Ug88efPLzWNY+OgK0WcE8ikWA2HBwvu7gyIK/XJTrPFFI6gJt9V/+xUEMcK1uLsJZMNkC2Tqu212K+1m0ABJ2VLeA6eltUAhcMjELiltRTpdOEq5EkYVTIYdh8vD16+5ClI799uLT5bsr2x0W6TqfiqJZ/qlve1++utRDNJedEHEi9XIb7p+mEkaFsH4N47W4yPM0d+wbYKs3QHSF1SMm7QNL5sSOCl7V5zzKi7Jr81B3wHy8St00E4kjRxpx3bGLC4sFYlYNzvP2+XUXlnIDTAYQvPtRdjpd5zkwW3OK3M7Wc5TVkD+uYNmRXeSrbhH9IZ743IAezML6iN0ylUumG+g3hxAG0sOK0xAF3CcQHkHfmog4TRYgw1KQPLNoPhcEHqOia92IXOLY6nuIWflw1q2mZED8xHca8/832/Uar3623YNAXhGHADAgvAqSEiAPQR6FsXgEQJLgG2vqGUAqwg4zqCmCKMnWZQD4LcV9WYHvzfJwXkpAuTcPW/sPMYSEYuYj/xbMwbp0ZN9cvA2uP9ljV3Y8L/dU/PTxp89GTUA0dIoMnCGaEZQRiAL6PAWaBVrBzlQFXfcQpt8zKdBMTXzPUmLCVQgzq7MTU00D119sxrI9AJVpTkHy/9j1bFa6AYJrD+IIBAx+dT0Nnc04DlCp9bknfgONZREKMXtgyDlVoeqEpsKtef08ftVuK4urpoAxWashLBGVmrtqIvKrJCWpJfeRkiQi+b4pIX59GZy/+Xz560VweXX9y+eKr3AJZJtDi8jkjxJetlhlscC1KtZT4IBivo7jbdfCtaa1PL/kVTyT616SRTERIDoFGipJlCzgJQgo1OdDK41noPZZ423CPAonsWCWxOE2YRzNLJp4ocjiT3MLTNaRMx0pWhqDZNtDTSQXVXWTsKBNRVoaZWZtk5KgdpuWzFY4P9KUFWxEXmO3GxXU1KnDYpBavfNWB+YwSiQoxaenXOtbkyf03CDQBwTNHuI9KIM/NfQnQBKbwmK6DJMFWo+VPC5A/aISPlO0lOUp0h+QU0NeyDlJ5gHBvhBBGAXUszMLy9CLkkl67/HYnrjPJPHgsjLUYEx+KnGQMAE7E+3WqIzBaF2jDauhBNuR4RxaidigvY3UV9myYOCXS7DmscePCbRXXZE539FSj9BnCTRRmTPQdOoiwiIAVxBbpXm0ABM8ps6SlgqTOLnLwY5PFh5hKgPDGgxCmARwsLgHykXWo7lbSDvgaMi5miIBbGlkaRLOTm6Pzjv/K+z8cdr5b+Pqq9UNOuPntkdykGZ7eLnf8ipfvqW1Ad+hANwhMslfIGCQsnDZ8xBQkmteL/Nt1XHTsIrS7o+IusuPtKzuTsuKJAaKCmlb4ffhbZTM1AtAHEy2Vl+WwJJG863j/uMAyA6RxmV34n4qstJyPn4iHHkVuqgTKH6Yf8CkQvMaJaZ1ffXOs378cI2s9D+uL9415TAglF1DNAslBKWkHv+LDc3tgd3NkoXt2dANPkxWGTxgZ/j0W7awvypCQfwpeVL18yiIHwGotvzI5CvXoHCcXVy6c6S5/ZaYE9joC9b9Ck73ulT2KL7d1eHXrnVxT+4V4nQBvCLQSZa+PFnLBN0QHWPgq+gP5KU7WKmQRYICe076zCe5TqzmDvlVd3U7i3LwitAALvzP+Vp4xJRBektPii5Kv8LoCBE9lh3nqCsR47LDRZxOYLpfmAO/Wu+ubqyz51+g9VfbwBAgEuobKgVlA74xXTPfRwIeSAkKpbwGyUzc+z32apbQ2kJADa9lPZ9H975t4yBc2e9ZIoYVmdvB5qW0WL5Q0Vdbt0P55vMsTpoT+MKdfuV56CY1QaB5kQQlsaJ9P7GJc0Aiz2Ix4I8uCkTBzFlrLSeK7fV7yZI/wTwvcGEKIqgBAX8MWGB9ssjTdVY4YJgUEsn4lQ0DemkKU3yhkDridUzXpT8aDwv/VC9rhIuaoyZxeh6IQu7nuFdfxMj3dRkyD34ZRaCu6UsxHkDPXaBGkcwcID2GBt67XuFFncIFN9mPTD0J9eWcANW5WIkgjjBSYEwtqYYcLnxz6uYsk8YU4TU2W7iv2W9egAcB//n+YtTp4bdqXhsALub3MMDIoedjatAfu2P41htAWb3bV2eNAdWTU3yfeLHLSPW4M5dkVLJeiRx0KbSu0QGahotRxGC93khwnQiQ/z0PxgX1VtB91erVvlavNu5YBU+WYTwHRnfAminqoRMHgyz8niOB8A0aI9RO3zvzXhivYUDf79dAwTLQ707/lb955b+gqW+InrDHnVXDzoR6P/V6jKrQoxd/RBnB4dFYvcHYdXWUhsKYAVnbTphPQfOHOZiiHhBbsKyMpjdYAUOL2k8AXJXLdJGC6dLBcKTsiuToNI3BKj5h2x+EaghN0JxTFgm8WsclaCcYrVwXoIXWCbxL442YgWYC+VWk4BXa8P3cHqBi8+wP6suP/IVVVmY6wQC/Jt+si9G2J34f1zRjlxb8XvkC1zJzX71U1jp0EhXzKEGZkrldRKZrlIGbFsZO5sHXHOO8UEdWMvQVrzpGIB2en8fz8G3sJawQBywsFoJNdECT1C+ZjxMpt5lAHq9UJa+H79OCDA6MAlgWnRDkm8WB4cI6sT51pms0FBOhTXIYuFhPZtEmQkVU9xrlYAOAZTTQ7EkRCQzxAAawf/+08r4Q+5PCmYFEAIwkW0Zb9bpbrFcONeqBKjqVdR6FNYkejlmXaLCn+YyU9w8YEk3YxOZIN4p75UTe+ygjAQKcrZMBb3mn4042OoUPlJXbdnmPy3tQXkPHvaKBYnsI86uIvBWDI+QSsAlVSsh+950Mxxm7z0fFvVdsxyYaCcG/KwS/0qg6SFdJZ5UCfMyXtCFxYv2Gke9qaRdbb3GvJPzvowHO0jMeT6VTGS+9eOMvtiyhoQ2J9H4loZevmHU2r/qHgCJZoLcmUqtYovFVpuAHTtHnyUWMoYCtcu22SF1ilZVbB7nyd9dD8h/m9zvfa9UagApIULgtYGm2o2JQHCdjP9lRfj/I7+vlq4hIGiEFFP4h8tRxAA4g0WdOjpaOJAOsB/+PHPjntR8voRi+vWJwOjFqgUqaddcZ+nTOOfBOvATcgftMtoPr/YivSOJAI+M9AhBAwaJcFv6IFqAK2mEDGM194vd2oZslJ6A7BKsGhKZCv7J+SVbLHaFeaz/I+iDRvyJmwEkCHQy57xHI2jGAvKpApbpLf7QBgqg0kaaieDlYIV0xPjb+KFf18lq91XFvoLGHlDc2xEwdjUA10R/oU8YBb3AFH/zlMUYuMZwdTc33G3xf41ytlpdajOtXm8PiZx9iMVwh18parYvSWqLRDp3PxBTrCMJ4LDaIaN6S+zCwXpycnfSt//zf/8f6z//4v/TJT716ZGMXALzPVgEwScEuRpLplGnnA5jDoC0LqwBHpphvKZQgdw9trdxR9uOU9yl2HVc5aASMlJoGG0VBpcwfqF04WQpEVLcs5Svt+RQUEHAKUTpgzHmxR44RkQg8SW/TMFW5PUurcc3YkX/H3M1PYLI+qh8kNrfGXFDRrURZzYjXUsQJvTj0lqHrORMvnnjLiasMKmjuYYANzCmABb+RLXycePI7kKr8hgavQXS08+pPjuNJJzT9AHr/2k84fAffX71sBBsqP4BMp7y0B6FnM1XaA2rj2bSM9uBB4ypKyCGbRrhvo4yTjlQj0oSwv3o2o1AaXXXDE1V0lKyFfgnk4EuUkw3Avk/ohccEnPt9YqymMR2if5qRH3o8H5+no+yfZcjU6C8nHo/gI6XSZP0WAUNtWCq3xmOVV8R8Ea5nETLdrLJyz+MY2LlkASrtDw5Zh1MCB2Naae4BDUxjaI4B7yjZnpgxL9yTNQzd2nYOPHozNPF83oBVOMRhvGUEjkw+XW796abfBWN09kYW0U6CtA2htzXM/gfXw1o3F59vgs83Fxf09Obn88ur4Pz6+ubjX4Orj1cXav6An1lh0vU08qZ190mBYdBpBjieIut4g/EQGd2X7F4a/ocUuFhQJ1Y5am2F5SA+DK+knEmblV2jrATTy7AMy6IYNrIO0MAtZI6B7U0m6T077ogWSs2AtboRCIKryELhIvLCOipwLu6fmgzTnx95z5+HI8mA48dBoYOzNICpmWojPsjNhDNNtLQtC4K6AA5WZLZGnpEJE05eCfVKT8sx25oJVC46C75MTfBYCTCHFiOlE8Ye60z9XqrQsVeBrMuMWYyrzTTFfAFjtCBdosCiYYNM5FPcL2dHrHd6+rwJxkmtUd/lLVB65BAW+ZFNTPvy0ytkXktQo7CAYsYRzIBUTtUcvbk8A2xRuowvd6E7izyaKZu8AANBJBS+J2dCJc4A3u/AihAduVmS5elsTba7hak4MNyUOu02NHq+jkWQkcUw81ZREmSRmAr/lL6jCD7V+h2Fkt+Ud+b6chVGnsIt0AShbHcJu2OUhEDYPO2eSugm6yhWS2cYcNpiY4N36Z/xl41/ZgQYVN4UesohmOXYQCGQ4tvlXYpGz7xDlhiG8ayLv755/8vby6t3ytL9IDdl1IgI7QKWI+EuImhDUkQZaB4GFoHyDetpiIoiW1OcOWR/Qfmd6GVaE7GIeK+zmmG8pRiMIAbXg8fb+j7MkoVB1cwdbmR4Ubb4EzYsvmJ0slvGGAV75kCo/jLhXVc5U43NQnWgpKuKeaMGGI3uwXgiIXFfhTPRa+DhXXc8vPfYqZZv0OPWqoa1ObZcVrBtj2XwWdUKqq65gTvA0OwQIFAiEODYSiufirY7y63vrA9mcgpZzWwrd3fAtGEHpILs/hGQ7R75qNa80VRjZ+MebC+50rAZoB57vkdH/13nsDmr8J72vn5w2Q4vxO9rkUzB4oWSAFhXhCoHpWHSJi8qqKq6tZg0VTszYtdEki8ku3t7WumWfTPqXa9sRrJfeO7z5MWxc4afZ8dOHz/7NTTwhg40qsKVUkooKdPyVmjAYhnNS79vDA6I0qLmQml6aw7aM0IrgxM8kb8L3n+Yg8NMDM22JrG3SvGcRUUWh1OBOZJS3twQvJzROcFETtxbwGlCFRAOZBGQdpYhQJIkgCuUopibCbIGxQ31pYbBeEkMygJtQlYcobVYo0ldCtw6BnbFvemtTJtg3YDdqFnVpY/aZ24ETDnakvmq2Ig58nqrAsNTb4UljZDqvyiHSm9H/pMDcnqcfyi8pnsxgpqtVlWQs9UsXjZCbOz3DTGkVou1PSKqpju9i2bl0luKaLEs/VWYoRTxfsceNFaoyuv+y79gJ1wTn8yuQCLq9ARJWx2mKjBM0oyHmmwDjlX6Riatg8pJRlXF72D+V1KGokaVpKnichQYwtoD3eWIAngb18NYMMi/sRIBcvPlO+stjINJyMB2AoMq81pmBEMd5luP5QvnSLxZism2WIpNgx8JEOzG1wmyjJfjnkfYAjnE+a9DqjbiBfN4fcZs3jHHgDEDfZc+qPXuTIIYUBANUwuC6QxkOnbhrUSZR1MfLf+imKRgWkgFirLA1+uJ0C81frTgLJZesvQROUuXdgmXNfeqWL4muActbxv7my9r4hykaLzsaBkI/S2P9VNLXCcbn7ruzJedYtnp1QqR5zaveu1h9dCb1tCb2tCbA0PTxDc+L00nARA2jfFrFKWJiXLfnGTjFUBQo/GOXvEPLMgZpdj4dZvU23jzpTdv7PoZM2aNo5sTrz3xHZMT3d0YMTpRhDOq+iESMx+B1LqINPe1xtHhfqc1og7i6FYw9blDLnpgNIOwm3+zQxQ+fYDEd8y/yVN/bqIglHLhz+M0LJ3HYhKZxz1ujUuvd4NJoxz73dPecweFPXBAvHSP6esGY/t7Z8f6vUrDpp5e4dvaBnjzb4VCD/xfBblSUseosh5qxR8t3cyvDc28B+RafIwbPSLGOyDp5dD8PG5Vs84IAU2EEFyEjJ7eI0+TQMYl6OyAJx8k72a+fFahJhlBrgVr6czBa52PUNEyRbMUTmn3m4bAfxgvDbuWUjKamz+ZX60LGhC0tyt3LRsSBqWiPF8RyG3hEWnDyD2uIp4aZNf9PjHIVRsOGP42cckTUvgCqqc1AL9bpvrNHE4E9MI5nncxjNwqVyWa+aO8iuZQTcz/pLCG9Ol3rXMrDYmHMpo2c46qXp5U1NJiahUaBIMJ8QnYQTlQGVfKCsUOb8dux0xHFXf4ynVf+Xqq+wgcAb+llbfPbcwTwBQBl/cEoB/y8hA5bpt4zMVg3FdbZAHa2g20Z3kq41N3ywiUC1J85Vf8hBkX+SSCPsAqn8YiTNbZyfv3H2TWa+GphG9YmzkHmTEjk9wC6An9DPBYEZ/SrbhQuZ0YLcMjYrA8RbEWBXD+FjpbhVHCWXAYo54C7jCHFagxJQNK3MujSvjBlpE+tBWhawITNVKyo2QucoxzyNNWcou9o8GTQcS6XxGaDoXEFptXk5qrQQiswt5AHKE6czCRJw7M9fG+GLHNVlasTM+VeYv85Id/m5jOAr+t2+uUnGf0rBN9qwzIxhrX4l3oBTqTZ6Z3Xq/NE9yX5dCamG5t8VjzKI41LHHoJSFp5jicCJBH3tUP7jCeeMnEeDvBt7zxBjY9FUzAaM63oDdjih46PEVdD6VLNCtwcdZJBAa9E4cjbDx2h6pMfo7k52upoYCdmk0nqqku48+R/FRNvwMfWdI+nVAs0w5iFNUKEjhlKymmyEPM0qR0TVhs9n1/wy1XErg1KpTtzQBHSucK1DQN0dUEPHwGM/d9qO8C/FiMcBswG/oH3po78a2VDBMDfIIb3RLUmXwmE8wEcMFKubQG5MoXIhUywwSx+i4l7pjKqi5AQFkB8pF6NYqf+PKJ/ESJf/cRZLgD+JXIFww3FAK685ACGBRottS05GxUW5+yqKLEQRqVoz9jlqTYcy0OrRpV0gBfV3sGNDS/MrcKHjEZNRMdbNEbeZQFBZoryqe4Xw9yEb6eqE0jm4azB/TxtYqXMTnJCVXjp3eJDzP1fagAjIOnYfzJs2/wuqYs8X1THODfQZv7GzV72ObGYMVsBEOOX501IhMtvMzFHZ1CgPXESDoLHMoL4UN8CY1Z5z25wHuEp4FptVOaKObtSDpCLZiTa6/SGRpoVgehcCkCIpCCtzX2hPHoQUDTwu+/PN2pkamKlOsBz9DU1LsCYTgiie7uNM3UwQAMnKrK8v2Q4oL+6bC4jbJqN80dhlNMMYYiKYjKNKC930NrLLveucxEO2pszqvTW1dqAJ7xgSEeHKECwt05CArtDiZUhNPSWGVU5TL2qOZt4ZKo0zK5WBfy4KXekKTYKsvssCwxrwvQg/JKHq2lJHTC7iu9wmbKxMOb5BRb/rs3ySs7YtbwlNmmCdoijOLSqtajd9XxL6v21WvOjtmZPISP/clt9x2xDnQIuZ7edt7pye11uM0OZKLHWPlyu31ZovyRHSVavqpckTFua+wdCfSx3xqqVetWbH2ncUwNjzTsPorm6Q6lN2uPq1fk2JpH5ow5wzDkIilK3I8gXaUbzmYOtGv3VwWbWtsTAPpjYKyknSHrdgJuuLqPWKQkrbY41LYCb3UwfA8sGcjWUOr1PI0xAbpTnwxAbiYE19oKislSYxgAUz8T5IgqkuPJ7ke6T5nINB6M9/cLAj9LMflgtJEyuQoSKm4bD1Wt0TQa+woUYDlSD4OxISeADs/67WFAXrKGwAAByxbyUTwtWtpkAMJllod3WiBRH54CxcNtDZJAP12+f3/x1hBl+ts+OOhfBXTlQzVooxGX4OFRSB1k/wf9Hernz/s85p8kx4a78wD1lSkmfyqZjPYxg/LEr5TWngAfTKttA7RmVhvgdbvBwbkoGDp8yAH1FTjis4UoH5jWd3gVS23PglmTL1qJ2dWVDvzHm8t3l1fn75VVVqapt6PDDExY9ObBj2LRoXcPOYrEDv0sB/mCdabr1ZrHURkKDdw5bL3sIAV6P2KK0LbDuIrsSirYiTjKB99LZLJkpDp99o1Gafd9mMQay6J2jkAlw2zT7CGaA5/aCwzvGrhH+83mHzA4qNtZzYeejKC2AtsdqirycyQ/X+8I9n5nYeIh6JeIL7eh9GKO7liv/TO145VjLAbWmS61EJlc1io0JP3l9pKCrxbujAwAwH9jMbFrks0gAXjM0NF+V9mgINNlPrRc4WoSLdbpGn2zaLGE5cqZisl5AMdG5LD22UOmBTlfiH2E8okv/ehdgBn+GBPe3+eFaRZYl4brdVAVq/lVLtGDguIGEwgIIUQR+tg33cbw5vMvIBiUdjHOwexY/2kJCgEIu2Uz1xTL32k1G9hV4V4z4l/p+2lSsrKeslnLYB1GW04oMBIb+aqZQsco2d4Ss8dZNOS67DDlCQ97p3QoIM8deEbHlTF3eGp36ZrC7Rt1G0BIgjwCboi3JLlJWD9mWt9Z14o2au73CWYX1kK96KNrSQMKYiFSpm9KLZntVActcUjn6Kq3WoLAe8rP0CXfqMGLZoNvsgUVHF5+pfeyNE4XWx33fWCVpVNaJc/KDTBeMnvQ9JnaShX/VCtaW/ShjE20iqof44Ls679mbgR4QwQ5vvagZu2ABScNHc+u2SwBMkGgVJ49qBV+bZMzBjSoUw5jEA6leYWxneEExPttM8AkawzqhbUYu1lYJdt7NIj3xca1zNAL5Sx/XFmYSRytohLMCUxngzKq/NrXvr+OEzU9biiQy8sV7YF6VsGkSVgIPP0RSHsTNV0zpKQyYhpbOG+Mq9asqvkAhDZJWbZepKIlBSID96FMUsVMfd67+bTC2zm4GNmQ91kwTxhAQ+MzHaAgj3LKIOabvdbJbQJKz7MKzOZepRvKYOWgSXV8cyamdHazShyrIG1kGE9jFCM6j7i2cbPjbELlWhDEps2AWxMBvaUU4mff4F+OkYa5jEQ536jkoEqlddihUA0zT91YZQSk9GDag/AJlH/Xg6soss50MizMSsLoVTfmrd55db/K6Kt1NRYdcVWlz3DSuwyAdv7ttYK+w2SxCotbVu3VjVkgWDYhk5a8UsVK8YYZnQxlBm67RjxYo+bffQWayZC62MikBMWQB2xBgL+oo4Xy6rhq17oWUJ1jMBAz3bunvRpLHd4cPScmPcnFb3xbCCW+W2pwSroE7iaw1nHJV0kJ8/ZIumBuKvTRGuRycx01+Lz5mMY16lZz4sJE3NUK9URr+5M4glp7jEHptYfmeutS1zGu2JCBCRB9SjTZA47Ht/Y06QKS1h4kUu4pb3030f4KN6FalHVRuZ+WqsmUNcFsjPJOiMTCUAbfEZIIneGtGWqX1MR3RmyqscIyxY1GJf+ckQEzer4Lco3XXYWvTweyI8wR6Hn8YBxd4QAAYJ5xXnP99XUCLXQ3NKyyloNNYahNeW6O6gTSh5fQ6BXBkZ/p+Tf2Wowl33Uimg09FKAVuhv8bNzzSIcAefQ/1T9ecmL0uyMuUSfsR/pATAK7XCDECBW3ffTWLt2jZgB+0WZH1mflu1fXDnxjGnjcHPZPAUuefYOJ/FdNQeS4sTgTIqvmgRulckPpVp5XbGWZYJwYwKg4rJZohcSngp3ow97KzGwNUR0kvRcF40nwaNiRkj5jM51emTsBXn9RLP/M5ttetfBAngwNVE/xlfeu8TUKqO/0fco6dZ5OAtHge3VA3Yb5/ypI//li8B8X1WiUs7CLknnq1zddGx1Uy14tetVrXT5zdhM6Kn/DXiTp0tGzL7Y8O2gP1OFBG69/puQX6bJeL9MyLZZppi33oV7+areeICDasI3LN7UvIL0z6RLoo9Cm+Ge3SkKrPKqqq+ZagE9COaiqwUqECflggI9otV61Fs8e7FpSY4RdGsbTIUteKUNVMaoZav3YhhsveASPXtV0KApp6kiPqpA/oSux5DNdy6pD7a8hlYz047jteXGFxtuxuTzsw325VVXpeTy6HZtpezo1QN25YKZ8uF+N/sCtWQOmxBxalwGsSLHODXVfHaBEx3K2lQW1LAZkAiZOKQRZ7rEvzVdJohjBIF2CpCMtVM4seP68ECVap4Vh2cZ34bbgc4HypOLSJGrq05Nb4yEpEgvv2PyJIvvqZtRHyjStdfhg5w75TXxswKnETXNCuz0k2b9yMf22m0D9y2rNPutxjp3qQcHFd9juqmJ7oELcdleskNrt6b3t9d2W2qnXbZTaxr6ROuranI899hUmqC6q3cFjWh2y/DEvbFKl+c2J99shSUaxawhTdTVKI7WF+Jlr121jr4V8FmiNykquGfOSAsXsuiFSDIniGwJFAt2SKTw9EAPGbP+cEGgcWja4QLKxvs93FcXAUyVyjvQZzfxrDqbUGUxV25G2usc1bKeutu/F1IBY+scQOBKLG3OtlFa6Ny2MwdvF0OGJuolSX5eaMZrDZxOF3eFcvps8+xZWLxNV8duk4STJO7dZC4WLXNDuIx4ktwd4jl5SR+j7kx1UQUokStdV/TI7ccrseA7/JWSYqgc2T6FKt9EcrFTB7fuyff95rYfqcWcf+p55rdGp7gHBr9LLMJE9oBuDjZva8ZuHQZjGRdsPXcvdpiHsRJFPQ37vuFl5Hy0ZsYQ2OSHs6mi6ZOw2FembtIGOQrylvnnlM/62RXe2XmWFulvaw6t5glux5atL3S6wAhhozR91IBJK7/jOvgXYb4Q8m1Gkh7MH+isYNgTs7rvRw30ZPW17TBpE4LJnSqdKIyZsyLuWKRQ2pFadGndZRGG9s6YtGCq+YNld3cgOqkYenKSL2T99Pn93EdxcXH+8+fyJfqtihHXHPqCwJsjgsX4V/H8pjeKCtW+NbwNHRVjxn0k96qaT9A5jaPAxalEP3t2uHnQlk4wO3mFeMeEe4jp4uTAz2BfEwle0jTh9EYCQt/upK8vRC6EfrOGfDInUT9w07iqvVnYRZq27hvbF3oeYgJAYW2VYgmn/myjlPfvaNT5ZNPOKuJ5yiG1xjzdIJxhqLXBL3PUaV84Wsc4fa+2ibU+9bY/8z1PofHQ67tLOVufMxftFHB0FhxJdnmbHZ5U5dX/q3Rs99Pb20Bvr8noPlL3iY8rB9nSw7Xn3p4P73lhmWTtU+vrUfcbfnviAh1rk/p+bdQ1zQP+P9OxsxKAiCOMuzs512+doZYtX/UG1jtXmpMy1HuBq2rKqdAaBdsCr5FdfG3qc4QSD8oeOTi44qU5FG9kiar4Ixxp9aGHjD2sA4d8FPHhADrNNl/NVIDaMcLsqsY2JeLa+MacQWZjTW31dTkMT85GkgHPuibopBgEzb/7qxcHtqaL/wE8CyV71bwL1d6jUjyr4JK8Qp0sZKAuLRCvu0Vl6c36vqm39KFB/JAfnHwQq8ulAXcrowwP/lIwEE1nBuKe2zhomp7lPfNXJQSl2ZWwrgOCifaWBpS+F/KK+fR3ilgrH23hNdt1zbtgu15fXF/bYPbFP+8ENrV7wkQXz5qX9525CJwupuiRcYkveEo6Xi8ufCsNr6ocbeS0yJ4iTiURd4g8WPbKj4Mvmq+xM3c5CxhuevXOc5xrN+DNa/ZcvqxTPYTiCwrE/wrd4WwUv28rn5UZfXx5uc4fRqluEG0FmRP2pixQS8OXnDt/EL01yjry1rmcaAtcXfl1tPBRT2/EjOAYd7voZHUUJNZOLAJJ77sDYCIg9wH/3mthGaw5AAEWxcVg3Ot6ff/qMIY/LTz9fXr2T1oddCzuhAi52tW0bLPVmzXvRApnVspUEYMsf0II51Q8YqXfmCSN8jzMe7ZSQY/ZLbD0C3+2FvzaghKAkOiVFpJu/gxLQmrJdvsk+IPPNsK94rb2IrpDx+5JiDIZUOL34K6JFIZSyndEp3iWaiDZJJgFtGnKENyUMYUvljZ8Oq3QgMDNXoBuItFyiiHgjB5l69sKa5qIA0j7GIBnvpLdqhyJK0A6Lg0lEd2sVrR9Hevr06UeAMgfX2qLkCSsWi3CKu0awPBYfNKhd2LgU+pcH7QKgwZ13MCjx0K4OwlU0qGIu8ipVX33x6lex+fVHz+Bpk7+92u2kfu1JXsos74thlHF88r52MOrAfW+MZYqNUIT/4f2dBrN9OP9rAAz602eKpxlbPzvqAcVdnEPFH3bE6niPqM3J2O7Nx5uri5vg4u0lMnP/5Wk7cNdueH3z8fPFm8/gQP/bz5efL+wdY7aiXXuG//n86h2GX2/w17M+XnG4saJu85fU/vXy+vriLTBVIxI+3MGHb365+fUiOP8FpqUbyHD3rr73ikRu29x9NFZ1tGfTY6wpBrkzqOS1aWbpPevHGGK7WaGiR7/66jXG9BvPHsz33cXHDxefb/5nILWWuk6yJh3kO/UbTyKeB6XhvC3SdFa/02V05vW9/thTn72e9y8yQlMA8ZfVLcjYlu4dlrcGazfqtrrH5kXjuoh5HGUUn2RlR0ZM8ywX9mueG1R/oHSw+SCj20ie+w2/oHFDHDj4GR+WiderBOMA01tn1Mn4RhPuo5HT3Jxg5t1+3/dBLsDHE3V4hs6Softl/PBENQO8ApDvIGQoqKbX8zomSzTHqd+pAz14+I/bwu0knJnIGcLzqNeH6WgzjDumTELdedheJN6pxKh5C9lV0cgxviO2fL/3F7zR3XyLv7MBZh589PdCUDXYAwhdY9o46bWzkSlBz2pEqTpqnYRoQCHayFAg1zWtqo6/+drpjdF70NOWrxr+Y9uIJG1d0BYFXRDpWT/gDr36KWBQo/2Ts5MXTE6kVMk96pDjxJlduF1u3pWn8YQFKsRf9ypRGcOYn7YJqGZMWlS2lYWMz1u+FB0Hbz3iW1Txh4Lxgs2Cj8DLZMf916d+Pfp/aM3AFw==')).decode()
V5_COMPAT_SOURCE=zlib.decompress(base64.b64decode('...')).decode()
V5_CORE=types.ModuleType("gnr_staircase_v5_embedded")
exec(V5_CORE_SOURCE,V5_CORE.__dict__)
V5_RUN_FINISHER_TRAINING=False
V5_USE_LEARNED_FINISHER=False
V5_RUN_LLM_REVIEW=False
V5_RUN_LEGACY_BATCH=False
V5_RUN_PAIR_MODEL_EXPERIMENTS=False
V5_RUN_EXTERNAL_CELLS=False
V5_ENABLE_GAP_REDRAW=False
V5_ENABLE_POCKET_FILL=False
V5_MAX_CHANGED_FRACTION=0.01  # Cumulative automatic edits vs Photoshop; not manual effort.
V5_PROTECTED_WHITE=None
print("V5.1 loaded: Photoshop baseline, broad fills OFF, 1% bounded automatic edit budget.")


In [ ]:
# V5 SELF-TEST — safe to run before mounting Drive. No GPU or API key required.
import json
print(json.dumps(V5_CORE.self_test(),indent=2))


In [ ]:
# ==================================================================================================
# CELL 1: SETUP  -  connect Google Drive, set the project folders, list the Step 2 / Step 3 files
# Stage_2 and Stage_3 are only READ. Everything new goes into GNR_Step3_Photoshop_Pipeline.
# ==================================================================================================
import re
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from google.colab import drive

drive.mount("/content/drive")

DATASET = Path("/content/drive/MyDrive/GNR Design Data-Set")      # Mahesh's shared folder (shortcut in My Drive)
STAGE2 = DATASET / "Stage_2"                                       # pencil sketches (read only)
STAGE3 = DATASET / "Stage_3"                                       # designer's production files (read only)
PIPE = DATASET / "GNR_Step3_Photoshop_Pipeline"                    # our work folder
INBOX = PIPE / "01_Photoshop_Inbox"                                # Photoshop exports go here
OUTPUT = PIPE / "02_Production_Output"                             # final BMP + PNG
REVIEW = PIPE / "03_Review"                                        # comparison images, logs

if not STAGE2.exists():
    raise FileNotFoundError("Cannot see " + str(STAGE2) + ". In Google Drive: Shared with me > GNR Design Data-Set "
                            "> right-click > Organize > Add shortcut > My Drive, then run this cell again.")
for folder in (INBOX, OUTPUT, REVIEW):
    folder.mkdir(parents=True, exist_ok=True)

NAME_PATTERN = re.compile(r"^\s*(.+?)\s+GNR\s*([23])\s*(\.(bmp|png|jpg|jpeg))?\s*$", re.IGNORECASE)


def pair_files(folder, step):
    """{design number: file} for names like '362 GNR 2.bmp' (extra spaces / missing extension are fine)."""
    found = {}
    for p in sorted(Path(folder).iterdir()):
        m = NAME_PATTERN.match(p.name)
        if p.is_file() and m and m.group(2) == str(step):
            found.setdefault(m.group(1).strip(), p)
    return found


def inbox_files():
    """{design number: file} for the Photoshop exports in the inbox ('362 GNR 3.png', '362.png' ...).
    Step 2 sketches ('... GNR 2 ...') are ignored; if a design has several exports, the newest one is used."""
    found = {}
    for p in sorted(INBOX.iterdir(), key=lambda q: q.stat().st_mtime):
        if not p.is_file() or p.suffix.lower() not in (".png", ".bmp", ".jpg", ".jpeg"):
            continue
        m = NAME_PATTERN.match(p.name)
        if m and m.group(2) == "2":
            continue                                                       # a Step 2 sketch, not an export
        found[p.stem.replace(chr(95), " ").split()[0]] = p                 # chr(95) is the underscore
    return found


S2 = pair_files(STAGE2, 2)
S3 = pair_files(STAGE3, 3)
both = sorted(set(S2) & set(S3), key=lambda k: (len(k), k))

print("=" * 90)
print("CELL 1: SETUP")
print("=" * 90)
print("Step 2 sketches        :", len(S2))
print("Designer Step 3 files  :", len(S3))
print("Designs with both      :", len(both), "  e.g.", both[:8])
print("Work folder            :", PIPE)
print("Photoshop inbox        :", sorted(inbox_files()) or "empty")
if "362" in S2:
    im = Image.open(S2["362"])
    print("Design 362 Step 2      :", S2["362"].name, im.size, im.mode)
print("Ready.", datetime.now().strftime("%d-%m-%Y %H:%M"))

In [ ]:
# V5 SHARED LIBRARY — new versioned file; original gnr_v4_lib.py is not changed.
V5_COMPAT_PATH=PIPE/"Notebook"/"gnr_v5_lib_94f8d65f8dd4.py"
V5_COMPAT_PATH.parent.mkdir(parents=True,exist_ok=True)
if V5_COMPAT_PATH.exists() and V5_COMPAT_PATH.read_text()!=V5_COMPAT_SOURCE:
    raise ValueError("Existing versioned v5 library differs; preserve it and use a new version.")
V5_COMPAT_PATH.write_text(V5_COMPAT_SOURCE)
print("Shared v5 batch library:",V5_COMPAT_PATH)


In [ ]:
V5_ACTIVE_INPUT = None  # A changed sketch invalidates older raster state.
# ==================================================================================================
# CELL 2: PHOTOSHOP STEP  -  pick a design; its Step 2 sketch (exact size, PNG) and the prompt are saved
# into Google Drive, so Photoshop can open them from Drive and export its result straight into the inbox.
# Photoshop gets ONLY the Step 2 sketch + this prompt (never the designer's Step 3).
# ==================================================================================================
DESIGN = "1071"                 # the design number to work on

if DESIGN not in S2:
    raise FileNotFoundError("No Step 2 sketch for design " + DESIGN + " in " + str(STAGE2))
sketch = Image.open(S2[DESIGN]).convert("RGB")
W, H = sketch.size

PHOTOSHOP_PROMPT = f"""Convert the supplied Step_2 sketch into a dense textile Step_3 design. Preserve motifs, positions, proportions, borders and meaningful white spaces. Ignore pencil/grid noise. Canvas: exactly {W} x {H} native pixels.
Use only pure red #FF0000 and white #FFFFFF. Fill intended solids red; preserve intentional white channels, cut-outs, veins and eyes. Keep distinct pieces separated by two native white pixels at their closest approach; preserve wider background spaces and genuinely continuous stems/curls.
Construct applicable curved corners from two flat anchors A and B through a shared one-pixel boundary transition M. Flat lengths are free. From either flat toward M, contour run lengths start at 2, 3 or 4, hold or decrease one level at a time, and end at 1. Example: horizontal 3,2,2 -> M(1 pixel) -> vertical 2,2,3; reading either half toward M gives 3,2,2,1. No skipping levels or growing toward M. There is no requirement about numbers of steps at each level. Apply in all orientations while preserving the shape; mirror only intended symmetry.
Keep equivalent repeated pieces consistent. Use crisp edges without antialiasing, bumps, stray pixels, gradients, textures or transparency. Preserve motif identity when uncertain. Export PNG/BMP at the original dimensions."""

# save the sketch (exact Step 2 size) and the prompt into Drive for Photoshop
PS_INPUT = PIPE / "00_Photoshop_Input"
PS_INPUT.mkdir(parents=True, exist_ok=True)
sketch_png = PS_INPUT / f"{DESIGN} GNR 2.png"
sketch.save(sketch_png)
prompt_txt = PS_INPUT / f"{DESIGN} prompt.txt"
prompt_txt.write_text(PHOTOSHOP_PROMPT, encoding="utf-8")
check = Image.open(sketch_png)

plt.figure(figsize=(9, 9 * H / W))
plt.imshow(sketch)
plt.title(f"Design {DESIGN} - Step 2 sketch ({W} x {H} px)")
plt.axis("off")
plt.show()

PC_ROOT = "G: (Google Drive) > My Drive > GNR Design Data-Set > GNR_Step3_Photoshop_Pipeline"
print("=" * 90)
print(f"CELL 2: PHOTOSHOP STEP  -  design {DESIGN}")
print("=" * 90)
print("Saved to Drive :", sketch_png.name, check.size, "+", prompt_txt.name)
print("               :", PS_INPUT)
print()
print("In Photoshop (on the PC with Google Drive):")
print("   Step A  File > Open:", PC_ROOT, "> 00_Photoshop_Input >", sketch_png.name)
print("           (check Image > Image Size shows", W, "x", H, "pixels)")
print("   Step B  AI Assisted mode: paste the prompt below (also saved as", prompt_txt.name + ") and generate.")
print("   Step C  File > Export > Export As > PNG at 100% size, AS IT IS (no hand edits, no Threshold), into:")
print("          ", PC_ROOT, "> 01_Photoshop_Inbox")
print("           named:", DESIGN, "GNR 3.png")
print("   Then run Cell 3.  (If Photoshop cannot reach Google Drive, save the PNG anywhere and use Cell 2B.)")
print()
print("-" * 30, "PROMPT (copy everything between the lines)", "-" * 30)
print(PHOTOSHOP_PROMPT)
print("-" * 104)

In [ ]:
V5_ACTIVE_INPUT = None  # Remains invalid if import or reporting fails.
V5_CORE.check_selected_sketch(globals())
# ==================================================================================================
# CELL 3: IMPORT + MEASURE  -  the Photoshop export at the exact Step 2 size in pure red / white
# (no cleaning yet), shown next to the designer's file, and measured against the designer's rules.
# The designer's Step 3 is used ONLY to compare, never as input.
# ==================================================================================================
from skimage.morphology import skeletonize

N8 = np.ones((3, 3), bool)
N4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], bool)


# ---------------- import ----------------
def redness_of(img):
    """0 = white ... 1 = full red (works for any shade of red, also dark red)."""
    a = np.asarray(img.convert("RGB")).astype(np.float32) / 255.0
    return np.clip(1.0 - (a[..., 1] + a[..., 2]) / 2.0, 0, 1)


def import_export(path, size):
    """Project palette at the exact native size; never silently resize geometry."""
    with Image.open(path) as opened:
        if opened.size != tuple(size):
            raise ValueError(f"Export size {opened.size} differs from Step_2 {tuple(size)}. Supply a registered native-size export; v5 refuses automatic resizing.")
        soft = redness_of(opened)
    return soft >= 0.5, soft, {"export_size": tuple(size), "note": "native size, no resize"}



def rgb_of(red):
    out = np.full(red.shape + (3,), 255, np.uint8)
    out[red] = (255, 0, 0)
    return out


# ---------------- measures ----------------
# Updated A/M/B geometry is loaded from the embedded v5 runtime.
V5_CORE.install_bindings(globals())



















def gap_widths(red):
    """% of separation-edge length with a white gap of 1 / 2 / 3 / 4 px to the nearest OTHER piece."""
    lab, n = ndi.label(red, structure=N8)
    out = {1: 0, 2: 0, 3: 0, 4: 0}
    for pid in range(1, n + 1):
        m = lab == pid
        ys, xs = np.nonzero(m)
        win = (slice(max(0, ys.min() - 6), ys.max() + 7), slice(max(0, xs.min() - 6), xs.max() + 7))
        other = (lab[win] != 0) & (lab[win] != pid)
        if not other.any():
            continue
        d = ndi.distance_transform_cdt(~other, metric="chessboard")
        edge = m[win] & ~ndi.binary_erosion(m[win], N4)
        for g in (1, 2, 3, 4):
            out[g] += int((edge & (d == g + 1)).sum())
    tot = max(1, sum(out.values()))
    return {g: round(100.0 * v / tot, 1) for g, v in out.items()}


def bump_count(red):
    """1-px bumps / nicks sticking out of a flat edge (red and white)."""
    total = 0
    for m in (red, ~red):
        Hm, Wm = m.shape
        p = np.pad(m, 2)
        at = lambda dy, dx: p[2 + dy:Hm + 2 + dy, 2 + dx:Wm + 2 + dx]
        nb = at(-1, 0).astype(int) + at(1, 0) + at(0, -1) + at(0, 1)
        hit = np.zeros_like(m)
        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            hit |= m & (nb == 1) & at(dy, dx) & at(dy + dx, dx + dy) & at(dy - dx, dx - dy)
        total += int(hit.sum())
    return total


def shape_match(m, ref):
    """Closeness to the designer: % of outline that lies within 1 px of the other (evaluation only)."""
    dr, dm = ndi.distance_transform_edt(~ref), ndi.distance_transform_edt(~m)
    p, r = float((dr[m] <= 1).mean()), float((dm[ref] <= 1).mean())
    return round(100 * 2 * p * r / max(1e-9, p + r), 1)


def measure(red, ref=None):
    lab, n = ndi.label(red, structure=N8)
    sizes = np.bincount(lab.ravel())[1:]
    wl, wn = ndi.label(~red, structure=N4)
    wsizes = np.bincount(wl.ravel())[1:]
    g = gap_widths(red)
    rep = {"red %": round(100 * float(red.mean()), 1), "pieces": n,
           "gaps 2 px % (target)": g[2], "gaps 1 px %": g[1], "gaps 3 px or wider %": round(g[3] + g[4], 1),
           "A/M/B corner screening %": curve_rule_pct(red), "bumps / nicks": bump_count(red),
           "red specks (1-3 px)": int((sizes <= 3).sum()), "pin-holes / broken lines": int((wsizes <= 2).sum())}
    if ref is not None:
        rep["shape match with designer %"] = shape_match(red, ref)
    return rep


# ---------------- run ----------------
# Optional: paste the FULL Drive path to an existing AI-generated Step_3 draft.
# Leave blank to reuse the inbox draft, or upload one when it is missing.
AI_DRAFT_PATH = ""
inbox = inbox_files()
if AI_DRAFT_PATH.strip():
    selected_path = Path(AI_DRAFT_PATH.strip()).expanduser()
    if not selected_path.is_file():
        raise FileNotFoundError(f"AI_DRAFT_PATH does not exist: {selected_path}")
    inbox[DESIGN] = V5_CORE.stage_ai_draft(selected_path.read_bytes(), INBOX, DESIGN, (W, H))
elif DESIGN not in inbox:
    print(f"Step_2 for design {DESIGN} is loaded ({W} x {H}).")
    print("This finisher needs an AI-generated Step_3 draft before it can edit the pixels.")
    print("Choose your Photoshop/Sunburst draft below, not the Step_2 sketch or designer reference.")
    print("If no draft exists yet, generate it using Cell 2's prompt, then rerun this cell.")
    from google.colab import files
    uploaded_drafts = files.upload()
    if len(uploaded_drafts) != 1:
        raise ValueError("Select exactly one AI Step_3 draft. You can also set AI_DRAFT_PATH above, then rerun this cell.")
    draft_bytes = next(iter(uploaded_drafts.values()))
    inbox[DESIGN] = V5_CORE.stage_ai_draft(draft_bytes, INBOX, DESIGN, (W, H))
print("Using AI draft:", inbox[DESIGN])
DESIGN_DIR = REVIEW / DESIGN
DESIGN_DIR.mkdir(parents=True, exist_ok=True)

RED_PS, SOFT_PS, import_info = import_export(inbox[DESIGN], (W, H))
REF = None
if DESIGN in S3:
    ref_img = Image.open(S3[DESIGN])
    if ref_img.size != (W, H):
        raise ValueError("Designer reference differs from Step_2 dimensions; register it explicitly before comparison.")
    REF = redness_of(ref_img) >= 0.5
Image.fromarray(rgb_of(RED_PS)).save(DESIGN_DIR / "1 photoshop pure.png")
np.save(DESIGN_DIR / "photoshop_soft.npy", SOFT_PS)

m_ps = measure(RED_PS, REF)
m_ref = measure(REF) if REF is not None else {}
soft_pct = round(100 * float(((SOFT_PS > 0.1) & (SOFT_PS < 0.9)).mean()), 1)

print("=" * 90)
print(f"CELL 3: IMPORT + MEASURE  -  design {DESIGN}")
print("=" * 90)
print("Export       :", inbox[DESIGN].name, import_info["export_size"], "-", import_info["note"])
print("Soft pixels  :", soft_pct, "% (anti-aliased edges; 0 means it was already pure red / white)")
print()
print(f"{'measure':34s}{'Photoshop':>12s}{'designer':>12s}")
for k, v in m_ps.items():
    print(f"{k:34s}{str(v):>12s}{str(m_ref.get(k, '')):>12s}")
print("(designer column = the designer's own Step 3, for comparison only)")

fig, ax = plt.subplots(1, 3 if REF is not None else 2, figsize=(21, 7 * H / W + 0.6))
ax[0].imshow(Image.open(inbox[DESIGN]).convert("RGB"))
ax[0].set_title("Photoshop export")
ax[1].imshow(rgb_of(RED_PS), interpolation="nearest")
ax[1].set_title(f"pure red / white, {W} x {H}")
if REF is not None:
    ax[2].imshow(rgb_of(REF), interpolation="nearest")
    ax[2].set_title("designer Step 3 (comparison only)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()
y0, x0 = H // 3, W // 3
zoom = (slice(y0, y0 + H // 3), slice(x0, x0 + W // 3))
fig, ax = plt.subplots(1, 2 if REF is not None else 1, figsize=(16, 8 * H / W + 0.6), squeeze=False)
ax[0, 0].imshow(rgb_of(RED_PS)[zoom], interpolation="nearest")
ax[0, 0].set_title("Photoshop (zoom, middle of the design)")
if REF is not None:
    ax[0, 1].imshow(rgb_of(REF)[zoom], interpolation="nearest")
    ax[0, 1].set_title("designer (same zoom)")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()
V5_ACTIVE_INPUT = V5_CORE.capture_input_context(globals(), inbox[DESIGN])
print("Input ready for design", DESIGN, "— run Cell 10 for the conservative final.")
V5_STAGE_REPORTS = {}
V5_CORE.record_stage(globals(), "Photoshop original", RED_PS)
print("For the conservative final, you can now run Cell 10 directly. Cells 4–6 are optional previews.")


In [ ]:
V5_CORE.require_input_context(globals())
# ==================================================================================================
# CELL 4: 2-PX SEPARATIONS  -  every white separation line between two pieces is redrawn exactly 2 px wide
# along its own centre (found on Photoshop's smooth drawing), like a designer's 2-px brush. Each line is
# checked on its own: if its redraw would merge or split pieces or add bumps, it is left as Photoshop drew it.
# ==================================================================================================
MAX_GAP = 5                # gaps up to this wide (px) count as separation lines
UP = 4                     # centre lines are found at 4x on the smooth drawing
MIN_THICK = 2.0            # a red stroke thinner than ~3 px is never cut into


def nearest_two(red, reach):
    """Per pixel: nearest piece (label, distance) and the distance to the nearest OTHER piece."""
    lab, n = ndi.label(red, structure=N8)
    Hh, Ww = red.shape
    d1 = np.full((Hh, Ww), np.inf)
    d2 = np.full((Hh, Ww), np.inf)
    l1 = np.zeros((Hh, Ww), int)
    for p, sl in enumerate(ndi.find_objects(lab), start=1):
        if sl is None:
            continue
        y0, y1 = max(0, sl[0].start - reach), min(Hh, sl[0].stop + reach)
        x0, x1 = max(0, sl[1].start - reach), min(Ww, sl[1].stop + reach)
        d = ndi.distance_transform_edt(lab[y0:y1, x0:x1] != p)
        D1, D2, L1 = d1[y0:y1, x0:x1], d2[y0:y1, x0:x1], l1[y0:y1, x0:x1]
        better = d < D1
        second = ~better & (d < D2)
        D2[better] = D1[better]
        D1[better], L1[better] = d[better], p
        D2[second] = d[second]
    return lab, l1, d1, d2


def bump_map(x):
    """1-px bumps sticking out of a flat edge."""
    Hm, Wm = x.shape
    p = np.pad(x, 2)
    at = lambda dy, dx: p[2 + dy:Hm + 2 + dy, 2 + dx:Wm + 2 + dx]
    nb = at(-1, 0).astype(int) + at(1, 0) + at(0, -1) + at(0, 1)
    hit = np.zeros_like(x)
    for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        hit |= x & (nb == 1) & at(dy, dx) & at(dy + dx, dx + dy) & at(dy - dx, dx - dy)
    return hit


def tidy(m, region):
    """Flip 1-px bumps / nicks inside `region` when that keeps the pieces and holes around them."""
    out = m.copy()
    for rnd in range(2):
        cand = (bump_map(out) | bump_map(~out)) & region
        if not cand.any():
            break
        for y, x in np.argwhere(cand):
            w = (slice(max(0, y - 3), y + 4), slice(max(0, x - 3), x + 4))
            before = (ndi.label(out[w], structure=N8)[1], ndi.label(~out[w], structure=N4)[1])
            out[y, x] = not out[y, x]
            if (ndi.label(out[w], structure=N8)[1], ndi.label(~out[w], structure=N4)[1]) != before:
                out[y, x] = not out[y, x]
    return out


def centre_points(soft, zone):
    """Sub-pixel centre-line points of the white lines inside `zone` (found at 4x on the smooth drawing)."""
    Hh, Ww = soft.shape
    s4 = cv2.resize(soft.astype(np.float32), (Ww * UP, Hh * UP), interpolation=cv2.INTER_CUBIC)
    z4 = cv2.resize(ndi.binary_dilation(zone, N8).astype(np.uint8), (Ww * UP, Hh * UP), interpolation=cv2.INTER_NEAREST) > 0
    sk = skeletonize((s4 < 0.5) & z4)
    ys, xs = np.nonzero(sk)
    pts = []
    for y, x in zip(ys, xs):
        ny, nx = np.nonzero(sk[max(0, y - 4):y + 5, max(0, x - 4):x + 5])
        if len(ny) >= 3:
            pts.append(((y + 0.5) / UP - 0.5, (x + 0.5) / UP - 0.5))
    return pts


def pen_mask(pts, shape):
    """A 2 x 2 px square pen stamped along the centre points - like a designer's 2-px brush."""
    Hh, Ww = shape
    m = np.zeros(shape, bool)
    for cy, cx in pts:
        r0, c0 = int(np.floor(cy)), int(np.floor(cx))
        m[max(0, r0):max(0, min(Hh, r0 + 2)), max(0, c0):max(0, min(Ww, c0 + 2))] = True
    return m


def two_px_separations(red, soft, protect=None):
    """Returns (new red, info)."""
    lab, l1, d1, d2 = nearest_two(red, MAX_GAP + 3)
    gap = ~red & (d1 + d2 <= MAX_GAP + 1.5) & np.isfinite(d2)
    if protect is not None:
        gap &= ~protect
    thick = ndi.distance_transform_edt(red)
    out = red.copy()
    comps, nc = ndi.label(gap, structure=N8)
    info = {"gaps": nc, "redrawn": 0, "left_as_is": 0}
    for k, sl in enumerate(ndi.find_objects(comps), start=1):
        y0, y1 = max(0, sl[0].start - 3), min(red.shape[0], sl[0].stop + 3)
        x0, x1 = max(0, sl[1].start - 3), min(red.shape[1], sl[1].stop + 3)
        g = comps[y0:y1, x0:x1] == k
        pts = centre_points(soft[y0:y1, x0:x1], g)
        if not pts:
            info["left_as_is"] += 1
            continue
        pen = pen_mask(pts, g.shape)
        pen &= g | (ndi.binary_dilation(g, N8) & (thick[y0:y1, x0:x1] >= MIN_THICK))
        before = out[y0:y1, x0:x1]
        after = before.copy()
        after[g] = True                                    # fill the old gap red ...
        after[pen] = False                                 # ... and draw the 2-px line
        changed = ndi.binary_dilation(after != before, N8)
        after = tidy(after, changed & ~pen)                # tidy 1-px bumps the redraw left (never the line)
        same_pieces = ndi.label(after, structure=N8)[1] == ndi.label(before, structure=N8)[1]
        if same_pieces and bump_count(after) <= bump_count(before):
            out[y0:y1, x0:x1] = after
            info["redrawn"] += 1
        else:
            info["left_as_is"] += 1
    # fill white specks (1-2 px) the redraw left inside red, unless that would join two pieces
    lab_old, n_old = ndi.label(~red, structure=N4)
    size_old = np.bincount(lab_old.ravel())
    size_old[0] = 99
    old_specks = ndi.binary_dilation((size_old <= 2)[lab_old] & ~red, N8)
    lab_new, n_new = ndi.label(~out, structure=N4)
    size_new = np.bincount(lab_new.ravel())
    size_new[0] = 99
    filled = 0
    for k in np.nonzero(size_new <= 2)[0]:
        sp = lab_new == k
        if (sp & old_specks).any():
            continue
        ys, xs = np.nonzero(sp)
        w = (slice(max(0, ys.min() - 3), ys.max() + 4), slice(max(0, xs.min() - 3), xs.max() + 4))
        trial = out[w].copy()
        trial[sp[w]] = True
        if ndi.label(trial, structure=N8)[1] == ndi.label(out[w], structure=N8)[1]:
            out[w] = trial
            filled += 1
    info["specks_filled"] = filled
    return out, info


# ---------------- run ----------------
if V5_ENABLE_GAP_REDRAW:
    proposed_sep, sep_info = two_px_separations(RED_PS, SOFT_PS)
    RED_SEP, sep_decision = V5_CORE.consider_proposal(RED_PS, RED_PS, proposed_sep,
        max_changed_fraction=V5_MAX_CHANGED_FRACTION, protected_white=globals().get('V5_PROTECTED_WHITE'))
    print("Experimental gap proposal:", sep_decision)
else:
    RED_SEP = RED_PS.copy()
    sep_info = {"gaps": 0, "redrawn": 0, "left_as_is": 0, "specks_filled": 0}
V5_CORE.record_stage(globals(), "Cell 4: gap redraw preview", RED_SEP)
Image.fromarray(rgb_of(RED_SEP)).save(DESIGN_DIR / "2 two-px separations.png")
m_sep = measure(RED_SEP, REF)

print("=" * 90)
print(f"CELL 4: 2-PX SEPARATIONS  -  design {DESIGN}")
print("=" * 90)
if V5_ENABLE_GAP_REDRAW:
    print("Separation lines :", sep_info["gaps"], "found,", sep_info["redrawn"], "redrawn at 2 px,",
          sep_info["left_as_is"], "left as drawn,", sep_info["specks_filled"], "stray white specks filled")
else:
    print("Gap redraw OFF: original Photoshop raster retained; no gaps redrawn.")
print("Pixels changed   :", int((RED_SEP != RED_PS).sum()), "of", W * H)
print()
print(f"{'measure':34s}{'Photoshop':>12s}{'2px lines':>12s}{'designer':>12s}")
for k, v in m_sep.items():
    print(f"{k:34s}{str(m_ps.get(k, '')):>12s}{str(v):>12s}{str(m_ref.get(k, '')):>12s}")
print("(gaps are measured corner to corner, so even 2-px lines on a slant partly count as 1 px;")
print(" the designer's own file is measured the same way)")

boxes = [(0, 0, W // 2, H // 2), (W // 2, H // 2, W - W // 2, H - H // 2), (W // 4, H // 4, W // 3, H // 3)]
for x0, y0, bw, bh in boxes:
    panels = [("Photoshop (pure)", RED_PS), ("2-px separations", RED_SEP)] + ([("designer", REF)] if REF is not None else [])
    fig, ax = plt.subplots(1, len(panels), figsize=(7 * len(panels), 7 * bh / bw + 0.5))
    for a, (t, m) in zip(ax, panels):
        a.imshow(rgb_of(m)[y0:y0 + bh, x0:x0 + bw], interpolation="nearest")
        a.set_title(f"{t}  x{x0}-{x0 + bw} y{y0}-{y0 + bh}")
        a.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
V5_CORE.require_input_context(globals())
# CELL 5: A/M/B CURVE CONSTRUCTION — updated to the supplied staircase diagram.
MAX_SHIFT = 2        # maximum native Chebyshev boundary displacement
MAX_REPEAT = 8       # bounded search, per 2/3/4 level; NOT a step-count hierarchy
V5_MAX_CORNER_EDITS = 250
V5_PROTECTED_WHITE = globals().get('V5_PROTECTED_WHITE', None)  # optional explicit native bool mask
SKIPPED = {}
V5_CORE.install_bindings(globals())
# constructor: V5_CORE.construct_corner(); validator: V5_CORE.corner_check().
# Unsupported junctions/ambiguous anchors remain unresolved for the designer.

# ---------------- run ----------------
V5_CORE.require_stage(globals(), "Cell 4: gap redraw preview", RED_SEP)
RED_CURVE, n_curves = fix_curves(RED_SEP)
V5_CORE.record_stage(globals(), "Cell 5: local staircase preview", RED_CURVE)
Image.fromarray(rgb_of(RED_CURVE)).save(DESIGN_DIR / "3 curve rule.png")
m_curve = measure(RED_CURVE, REF)

print("=" * 90)
print(f"CELL 5: CURVE RULE  -  design {DESIGN}")
print("=" * 90)
print("Curves rebuilt   :", n_curves, "  (pixels changed:", int((RED_CURVE != RED_SEP).sum()), ")")
print("Left as they are :", ", ".join(f"{k} {v}" for k, v in sorted(SKIPPED.items(), key=lambda kv: -kv[1])))
print()
print(f"{'measure':34s}{'after Cell 4':>14s}{'curve rule':>12s}{'designer':>12s}")
for k, v in m_curve.items():
    print(f"{k:34s}{str(m_sep.get(k, '')):>14s}{str(v):>12s}{str(m_ref.get(k, '')):>12s}")

boxes = [(0, 0, W // 2, H // 2), (W // 2, H // 2, W - W // 2, H - H // 2), (W // 4, H // 4, W // 3, H // 3)]
for x0, y0, bw, bh in boxes:
    panels = [("after Cell 4 (2-px lines)", RED_SEP), ("+ curve rule", RED_CURVE)] + ([("designer", REF)] if REF is not None else [])
    fig, ax = plt.subplots(1, len(panels), figsize=(7 * len(panels), 7 * bh / bw + 0.5))
    for a, (t, m) in zip(ax, panels):
        a.imshow(rgb_of(m)[y0:y0 + bh, x0:x0 + bw], interpolation="nearest")
        a.set_title(f"{t}  x{x0}-{x0 + bw} y{y0}-{y0 + bh}")
        a.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
V5_CORE.require_input_context(globals())
# ==================================================================================================
# CELL 6: PETAL GROUPS  -  like the designer, neighbouring pieces of a group (petals, leaves of a fan) grow
# into the white wedges between them and meet along one even 2-px line, so the group reads as one clean
# shape. Only white that lies BETWEEN two different pieces is filled; curl interiors, eyes and open
# background stay white; no two pieces ever join. Then the curve rule (Cell 5) runs again on the new outlines.
# Needs Cells 3, 4 and 5 to have run in this session.
# ==================================================================================================
POCKET = 3                 # wedges / pockets up to about 2 x 3 = 6 px wide between pieces are filled
ISLAND = 40                # small white islands where 3+ pieces meet are filled too


def disc(r):
    yy, xx = np.ogrid[-r:r + 1, -r:r + 1]
    return (yy * yy + xx * xx) <= r * r + 0.5


def fill_pockets(red, R):
    """Returns (new red, info)."""
    Hh, Ww = red.shape
    lab, n = ndi.label(red, structure=N8)
    pad = R + 2
    closed = ndi.binary_closing(np.pad(red, pad), structure=disc(R))[pad:-pad, pad:-pad]
    pocket = closed & ~red
    # enclosed holes (white areas touching only one piece: eyes, dot rings, cut-outs) are never filled
    wl, wn = ndi.label(~red, structure=N4)
    hole = np.zeros(wn + 1, bool)
    for k, sl in enumerate(ndi.find_objects(wl), start=1):
        if sl is None:
            continue
        y0, y1 = max(0, sl[0].start - 1), min(Hh, sl[0].stop + 1)
        x0, x1 = max(0, sl[1].start - 1), min(Ww, sl[1].stop + 1)
        ring = ndi.binary_dilation(wl[y0:y1, x0:x1] == k, N8) & (lab[y0:y1, x0:x1] > 0)
        touches = np.unique(lab[y0:y1, x0:x1][ring])
        border = sl[0].start == 0 or sl[1].start == 0 or sl[0].stop == Hh or sl[1].stop == Ww
        hole[k] = len(touches) <= 1 and not border
    pocket &= ~hole[wl]
    # nearest piece for every pixel, and the distance to the nearest OTHER piece
    dist, (iy, ix) = ndi.distance_transform_edt(~red, return_indices=True)
    near = lab[iy, ix]
    d2 = np.full((Hh, Ww), np.inf)
    for p, sl in enumerate(ndi.find_objects(lab), start=1):
        if sl is None:
            continue
        y0, y1 = max(0, sl[0].start - 2 * R - 2), min(Hh, sl[0].stop + 2 * R + 2)
        x0, x1 = max(0, sl[1].start - 2 * R - 2), min(Ww, sl[1].stop + 2 * R + 2)
        d = ndi.distance_transform_edt(lab[y0:y1, x0:x1] != p)
        sub = d2[y0:y1, x0:x1]
        upd = (near[y0:y1, x0:x1] != p) & (d < sub)
        sub[upd] = d[upd]
    pocket &= d2 <= 2 * R + 1                       # only white BETWEEN two different pieces
    for rnd in range(2):                            # small islands where 3+ pieces meet
        rl, rn = ndi.label(~red & ~pocket, structure=N4)
        size = np.bincount(rl.ravel())
        for k, sl in enumerate(ndi.find_objects(rl), start=1):
            if sl is None or size[k] > ISLAND:
                continue
            isl = rl == k
            if not (ndi.binary_dilation(isl, N8) & pocket).any():
                continue
            if np.unique(near[isl]).size >= 2 or d2[isl].min() <= 2 * R + 3:
                pocket |= isl & ~hole[wl]
    for rnd in range(2):                            # smooth the nearest-piece map -> straighter lines
        keys = list(np.unique(near[pocket]))
        if keys:
            stack = np.stack([ndi.uniform_filter((near == k).astype(np.float32), size=3) for k in keys])
            near = np.where(pocket, np.asarray(keys)[stack.argmax(0)], near)
    # the meeting line: pixels with a side neighbour of another piece -> 2 px per row / column
    nl = np.where(red | pocket, near, 0)
    p4 = np.pad(nl, 1)
    line = np.zeros_like(red)
    for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        q = p4[1 + dy:Hh + 1 + dy, 1 + dx:Ww + 1 + dx]
        line |= (q > 0) & (q != nl) & (nl > 0)
    out = red | (pocket & ~line)
    for rnd in range(3):                            # two pieces may not touch at a pixel corner
        own = np.where(out, near, 0)
        fixed_any = False
        for a, b, oa, ob in ((own[:-1, :-1], own[1:, 1:], (0, 0), (1, 1)),
                             (own[:-1, 1:], own[1:, :-1], (0, 1), (1, 0))):
            for y, x in np.argwhere((a > 0) & (b > 0) & (a != b)):
                for yy, xx in ((y + oa[0], x + oa[1]), (y + ob[0], x + ob[1])):
                    if out[yy, xx] and not red[yy, xx]:
                        out[yy, xx] = False
                        fixed_any = True
                        break
        if not fixed_any:
            break
    lab_n, nn = ndi.label(out, structure=N8)        # filled bits cut off from their own piece are dropped
    has_old = np.zeros(nn + 1, bool)
    has_old[np.unique(lab_n[red])] = True
    out &= has_old[lab_n] | red
    region = ndi.binary_dilation(pocket, N8)        # tidy 1-px bumps / nicks inside the filled area
    for rnd in range(2):
        for y, x in np.argwhere((bump_map(out) | bump_map(~out)) & region):
            w = (slice(max(0, y - 3), y + 4), slice(max(0, x - 3), x + 4))
            before = (ndi.label(out[w], structure=N8)[1], ndi.label(~out[w], structure=N4)[1])
            out[y, x] = not out[y, x]
            if (ndi.label(out[w], structure=N8)[1], ndi.label(~out[w], structure=N4)[1]) != before:
                out[y, x] = not out[y, x]
    lab2, n2 = ndi.label(out, structure=N8)         # guard: pieces never join
    if n2 < n:
        for k in range(1, n2 + 1):
            if np.unique(lab[(lab2 == k) & red]).size > 1:
                out[(lab2 == k) & ~red] = False
        n2 = ndi.label(out, structure=N8)[1]
    return out, {"filled_px": int((out & ~red).sum()), "pieces": (n, n2)}


def gap_rows(red):
    """Separation width as the designer counts it: white pixels per row / column across a line between two
    different pieces. Returns % of line cross-sections that are 1 / 2 / 3 / 4 px."""
    lab = ndi.label(red, structure=N8)[0]
    Hh, Ww = red.shape
    hl = np.full((Hh, Ww), 99)
    vl = np.full((Hh, Ww), 99)
    for arr, L in ((lab, hl), (lab.T, vl.T)):
        for i in range(arr.shape[0]):
            row = arr[i]
            j = 0
            while j < len(row):
                if row[j] == 0:
                    k = j
                    while k + 1 < len(row) and row[k + 1] == 0:
                        k += 1
                    if j > 0 and k + 1 < len(row) and row[j - 1] != row[k + 1] and k - j + 1 <= 6:
                        L[i, j:k + 1] = k - j + 1
                    j = k + 1
                else:
                    j += 1
    wmin = np.minimum(hl, vl)
    cnt = {g: float(((wmin == g) / g).sum()) for g in (1, 2, 3, 4)}
    tot = max(1e-9, sum(cnt.values()))
    return {g: round(100 * v / tot, 1) for g, v in cnt.items()}


# ---------------- run ----------------
V5_CORE.require_stage(globals(), "Cell 5: local staircase preview", RED_CURVE)
if V5_ENABLE_POCKET_FILL:
    proposed_petal, petal_info = fill_pockets(RED_CURVE, POCKET)
    RED_FINAL, pocket_decision = V5_CORE.consider_proposal(RED_PS, RED_CURVE, proposed_petal,
        max_changed_fraction=V5_MAX_CHANGED_FRACTION, protected_white=globals().get('V5_PROTECTED_WHITE'))
    print("Experimental pocket proposal:", pocket_decision)
else:
    RED_FINAL = RED_CURVE.copy()
    petal_info = {"filled_px": 0, "pieces": (ndi.label(RED_CURVE, N8)[1], ndi.label(RED_CURVE, N8)[1])}
    print("Pocket fill OFF: no white wedges, channels or holes filled.")
RED_PETAL = RED_FINAL.copy()
n1 = n2 = 0
V5_CORE.record_stage(globals(), "Cell 6: pocket fill preview", RED_FINAL)
Image.fromarray(rgb_of(RED_FINAL)).save(DESIGN_DIR / "4 petal groups + curve rule.png")
m_final = measure(RED_FINAL, REF)

print("=" * 100)
print(f"CELL 6: PETAL GROUPS  -  design {DESIGN}")
print("=" * 100)
print("Wedges filled    :", petal_info["filled_px"], "px   pieces", petal_info["pieces"][0], "->", petal_info["pieces"][1],
      "  | curves rebuilt again:", n1 + n2)
print()
cols = [("Photoshop", RED_PS, m_ps), ("Cell 4", RED_SEP, m_sep), ("Cell 5", RED_CURVE, m_curve),
        ("Cell 6", RED_FINAL, m_final)] + ([("designer", REF, m_ref)] if REF is not None else [])
print(f"{'measure':40s}" + "".join(f"{c[0]:>11s}" for c in cols))
rows_g = [gap_rows(c[1]) for c in cols]
for g, label in ((2, "lines 2 px per row/col % (target)"), (1, "lines 1 px per row/col %"), (3, "lines 3 px per row/col %"),
                 (4, "lines 4 px per row/col %")):
    print(f"{label:40s}" + "".join(f"{str(r[g]):>11s}" for r in rows_g))
for k in ("red %", "pieces", "A/M/B corner screening %", "bumps / nicks", "shape match with designer %"):
    print(f"{k:40s}" + "".join(f"{str(c[2].get(k, '')):>11s}" for c in cols))

x0, y0, bw, bh = 100, 80, 133, 106             # the petal area you marked
panels = [("Cell 5", RED_CURVE), ("Cell 6: petal groups", RED_FINAL)] + ([("designer", REF)] if REF is not None else [])
fig, ax = plt.subplots(1, len(panels), figsize=(7 * len(panels), 7 * bh / bw + 0.5))
for a, (t, m) in zip(ax, panels):
    a.imshow(rgb_of(m)[y0:y0 + bh, x0:x0 + bw], interpolation="nearest")
    a.set_title(f"{t}  x{x0}-{x0 + bw} y{y0}-{y0 + bh}")
    a.axis("off")
plt.tight_layout()
plt.show()
for x0, y0, bw, bh in ((0, 0, W // 2, H // 2), (W // 2, H // 2, W - W // 2, H - H // 2)):
    fig, ax = plt.subplots(1, len(panels), figsize=(7 * len(panels), 7 * bh / bw + 0.5))
    for a, (t, m) in zip(ax, panels):
        a.imshow(rgb_of(m)[y0:y0 + bh, x0:x0 + bw], interpolation="nearest")
        a.set_title(f"{t}  x{x0}-{x0 + bw} y{y0}-{y0 + bh}")
        a.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
if V5_RUN_FINISHER_TRAINING:
    # ==================================================================================================
    # CELL 7: TRAINING DATA  -  the designer's own Step 3 files teach the finishing. Each training example is a
    # crop of a designer file roughened to look like a Photoshop AI export (rounded blobs, white wedges where
    # petals meet, ~1 px wider separations, soft wobbly edges) paired with the designer's original pixels.
    # Test designs (1071, 362, the current DESIGN and the 10 Batch_01 designs) are kept OUT of training so the
    # test is honest.
    # v2: thinning calibrated on real Photoshop exports (they lose 5-9 points of red, not ~10), and half of the
    # examples are pure red / white like Photoshop's newer exports.
    # Needs Cells 1 and 3 (and a GPU runtime for Cell 8: Runtime > Change runtime type > A100, High-RAM).
    # ==================================================================================================
    import os
    import multiprocessing as mp
    from concurrent.futures import ThreadPoolExecutor

    BATCH_01 = ["828", "261", "827", "694", "371", "752", "1433", "95", "1191", "794"]
    HOLD_OUT = {"1071", "362", DESIGN} | set(BATCH_01)      # never used for training
    CROP = 128                              # training crops are CROP x CROP pixels
    N_TRAIN = 40000                         # number of training crops
    N_VAL = 2000                            # number of validation crops (from designs not used for training)
    VAL_SHARE = 0.05                        # share of designs kept apart for validation
    UP = 3                                  # roughening is done on a 3x canvas
    SHRINK = (0.25, 1.0)                    # px per side the Photoshop drawing is thinner than the designer's
    HARD_SHARE = 0.5                        # share of examples given as pure red / white (no soft edges)
    WOBBLE = (0.05, 0.25)                   # px of low-frequency edge wobble


    def disc(r):
        r = max(1, int(round(r)))
        yy, xx = np.ogrid[-r:r + 1, -r:r + 1]
        return (yy * yy + xx * xx) <= r * r + 0.5


    def smooth_noise(shape, scale, rng):
        """Low-frequency noise in [-1, 1] with features about `scale` px."""
        h, w = shape
        n = rng.standard_normal((max(2, int(h / scale) + 2), max(2, int(w / scale) + 2))).astype(np.float32)
        return np.clip(cv2.resize(n, (w, h), interpolation=cv2.INTER_CUBIC) / 1.5, -1, 1)


    def roughen(T, rng):
        """Designer mask (bool) -> Photoshop-like soft redness (0..1) of the same size."""
        h, w = T.shape
        big = cv2.resize(T.astype(np.uint8), (w * UP, h * UP), interpolation=cv2.INTER_NEAREST).astype(bool)
        s = cv2.GaussianBlur(big.astype(np.float32), (0, 0), UP * 0.55) >= 0.5          # the smooth 'ideal' curve
        r = rng.uniform(1.5, 3.2) * UP                                                     # rounded corners of thick shapes
        opened = ndi.binary_opening(s, structure=disc(r))
        inside = ndi.distance_transform_edt(s)
        s = np.where(ndi.maximum_filter(inside, size=int(2 * r + 1)) >= r + 0.5, opened, s)
        if rng.random() < 0.15:                                                            # a separation sometimes missing
            closed = ndi.binary_closing(s, structure=disc(UP * 1.2))
            s = np.where(smooth_noise(s.shape, 40 * UP, rng) > 0.6, closed, s)
        din = ndi.distance_transform_edt(s)
        sdf = din - ndi.distance_transform_edt(~s)
        thin_factor = np.clip((ndi.maximum_filter(din, size=4 * UP + 1) / UP - 1.0) / 1.5, 0.0, 1.0)
        shrink_px = (rng.uniform(*SHRINK) + 0.25 * smooth_noise(s.shape, 30 * UP, rng)) * thin_factor
        wob = rng.uniform(*WOBBLE) * smooth_noise(s.shape, rng.uniform(4, 9) * UP, rng) * (0.3 + 0.7 * thin_factor)
        sdf = sdf - UP * shrink_px + UP * wob
        width = rng.uniform(0.5, 0.9) * UP
        soft = cv2.resize(np.clip(0.5 + sdf / (2 * width), 0, 1).astype(np.float32), (w, h), interpolation=cv2.INTER_AREA)
        lab, n = ndi.label(soft >= 0.5, structure=N8)                                      # fade broken-off specks
        if n:
            speck = (np.bincount(lab.ravel()) <= 6)[lab] & (lab > 0)
            soft[speck] = np.minimum(soft[speck], 0.45)
        if rng.random() < HARD_SHARE:                                                      # exported as pure red / white
            soft = (soft >= 0.5).astype(np.float32)
        return soft


    def make_example(args):
        """One (input, target) crop: args = (design index, seed). Uses the global DESIGNS list."""
        i, seed = args
        rng = np.random.default_rng(seed)
        T = DESIGNS[i]
        margin = 8
        full = CROP + 2 * margin
        if T.shape[0] < full or T.shape[1] < full:
            T = np.pad(T, ((0, max(0, full - T.shape[0])), (0, max(0, full - T.shape[1]))))
        for attempt in range(10):
            y, x = rng.integers(0, T.shape[0] - full + 1), rng.integers(0, T.shape[1] - full + 1)
            c = T[y:y + full, x:x + full]
            if 0.15 < c.mean() < 0.9:
                break
        soft = roughen(c, rng)
        sl = slice(margin, margin + CROP)
        return (np.round(soft[sl, sl] * 255).astype(np.uint8), c[sl, sl].astype(np.uint8))


    def load_design(p):
        try:
            return redness_of(Image.open(p)) >= 0.5
        except Exception:
            return None


    # ---------------- run ----------------
    t0 = datetime.now()
    ids = sorted(k for k in S3 if k not in HOLD_OUT)
    with ThreadPoolExecutor(16) as ex:
        loaded = list(ex.map(load_design, [S3[k] for k in ids]))
    keep = [(k, m) for k, m in zip(ids, loaded) if m is not None and min(m.shape) >= 32 and 0.05 < m.mean() < 0.95]
    rng0 = np.random.default_rng(7)
    order = rng0.permutation(len(keep))
    n_val = max(1, int(len(keep) * VAL_SHARE))
    val_ids = [keep[i][0] for i in order[:n_val]]
    train_ids = [keep[i][0] for i in order[n_val:]]
    DESIGNS = [keep[i][1] for i in order]                      # validation designs first, then training designs
    areas = np.array([m.size for m in DESIGNS], float)


    def jobs(index_range, n, seed):
        rng = np.random.default_rng(seed)
        idx = np.asarray(index_range)
        p = areas[idx] / areas[idx].sum()                      # bigger designs give more crops
        picks = rng.choice(idx, size=n, p=p)
        return [(int(i), int(s)) for i, s in zip(picks, rng.integers(0, 2147483647, size=n))]


    try:
        ctx = mp.get_context("fork")                           # Colab (Linux): all CPU cores in parallel
    except ValueError:
        ctx = None
    if ctx is not None:
        with ctx.Pool(os.cpu_count()) as pool:
            tr = pool.map(make_example, jobs(range(n_val, len(DESIGNS)), N_TRAIN, 1), chunksize=64)
            va = pool.map(make_example, jobs(range(0, n_val), N_VAL, 2), chunksize=64)
    else:
        tr = list(map(make_example, jobs(range(n_val, len(DESIGNS)), N_TRAIN, 1)))
        va = list(map(make_example, jobs(range(0, n_val), N_VAL, 2)))
    X_TRAIN = np.stack([a for a, b in tr])
    Y_TRAIN = np.stack([b for a, b in tr])
    X_VAL = np.stack([a for a, b in va])
    Y_VAL = np.stack([b for a, b in va])
    del tr, va

    print("=" * 90)
    print("CELL 7: TRAINING DATA")
    print("=" * 90)
    print("Designer files used :", len(train_ids), "for training,", len(val_ids), "for validation;  kept out:", sorted(HOLD_OUT))
    print("Training crops      :", X_TRAIN.shape, "  validation crops:", X_VAL.shape)
    print("Changed pixels      :", round(100 * float(((X_TRAIN[:2000] >= 128) != Y_TRAIN[:2000]).mean()), 2),
          "% of pixels differ between the roughened input and the designer (what the model must learn to fix)")
    print("Time                :", str(datetime.now() - t0).split(".")[0])
    fig, ax = plt.subplots(2, 6, figsize=(18, 6.4))
    for j in range(6):
        ax[0, j].imshow(np.dstack([np.full((CROP, CROP), 255), 255 - X_TRAIN[j], 255 - X_TRAIN[j]]).astype(np.uint8), interpolation="nearest")
        ax[0, j].set_title("roughened (input)")
        ax[1, j].imshow(rgb_of(Y_TRAIN[j].astype(bool)), interpolation="nearest")
        ax[1, j].set_title("designer (target)")
    for a in ax.ravel():
        a.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print('Synthetic training-data generation skipped.')


In [ ]:
if V5_RUN_FINISHER_TRAINING:
    # ==================================================================================================
    # CELL 8: TRAIN THE FINISHER  -  a small U-Net learns to turn a Photoshop-like drawing into the designer's
    # pixels (2-px lines, staircase edges, petals packed into their groups). Runs on the GPU (A100: ~10-15 min).
    # The trained model is saved to Drive: GNR_Step3_Photoshop_Pipeline / 05_Finisher_Model / MODEL_NAME
    # (a new name for every version; earlier models are kept).
    # ==================================================================================================
    import time
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    STEPS = 15000               # training steps (A100: about 8 minutes)
    BATCH = 64                  # crops per step
    LR = 2e-3                   # learning rate (cosine decay to ~0)
    EDGE_WEIGHT = 5.0           # pixels near an edge count this much more (that is where the work is)
    MODEL_NAME = "finisher_v2.pt"
    MODEL_DIR = PIPE / "05_Finisher_Model"
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


    def block(cin, cout):
        return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                             nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


    def make_unet(ch=(32, 64, 128, 256, 384)):
        """U-Net. Input: Photoshop redness (1 channel). Output: designer red / white (1 channel, logits)."""
        m = nn.ModuleDict()
        m["downs"] = nn.ModuleList([block(a, b) for a, b in zip((1,) + ch[:-1], ch)])
        m["upconv"] = nn.ModuleList([nn.ConvTranspose2d(a, b, 2, stride=2) for a, b in zip(ch[:0:-1], ch[-2::-1])])
        m["ups"] = nn.ModuleList([block(2 * b, b) for b in ch[-2::-1]])
        m["head"] = nn.Conv2d(ch[0], 1, 1)
        return m


    def run_unet(m, x):
        skips = []
        for i, d in enumerate(m["downs"]):
            x = d(x)
            if i < len(m["downs"]) - 1:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, blk in zip(m["upconv"], m["ups"]):
            x = up(x)
            x = blk(torch.cat([x, skips.pop()], 1))
        return m["head"](x)


    def augment(x, y):
        """Same random flip / quarter turn for input and target (the rules look the same in every direction)."""
        k = int(torch.randint(0, 4, (1,)))
        x, y = torch.rot90(x, k, (2, 3)), torch.rot90(y, k, (2, 3))
        if torch.rand(1) < 0.5:
            x, y = torch.flip(x, (3,)), torch.flip(y, (3,))
        return x, y


    def edge_weights(x, y):
        """1 everywhere, EDGE_WEIGHT within 2 px of a designer edge or where input and target disagree."""
        edge = (F.max_pool2d(y, 5, 1, 2) - (-F.max_pool2d(-y, 5, 1, 2))) > 0
        differ = (x >= 0.5).float() != y
        return 1.0 + (EDGE_WEIGHT - 1.0) * (edge | differ).float()


    def evaluate(model, X, Y, n=2000):
        """Validation: % of pixels wrong; % of the needed fixes made; % of pixels changed that should not have been."""
        model.eval()
        wrong = total = fix_ok = fix_total = bad = keep_total = 0
        with torch.no_grad():
            for i in range(0, min(n, len(X)), 256):
                x = torch.from_numpy(X[i:i + 256]).float().div(255).unsqueeze(1).to(DEVICE)
                y = torch.from_numpy(Y[i:i + 256]).float().unsqueeze(1).to(DEVICE)
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
                    p = run_unet(model, x)
                pred = (p.float() > 0).float()
                need = (x >= 0.5).float() != y
                wrong += int((pred != y).sum())
                total += y.numel()
                fix_ok += int(((pred == y) & need).sum())
                fix_total += int(need.sum())
                bad += int(((pred != y) & ~need).sum())
                keep_total += int((~need).sum())
        model.train()
        return 100.0 * wrong / total, 100.0 * fix_ok / max(1, fix_total), 100.0 * bad / max(1, keep_total)


    # ---------------- run ----------------
    print("=" * 90)
    print("CELL 8: TRAIN THE FINISHER  -  device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
    print("=" * 90)
    if DEVICE == "cpu":
        print("No GPU found - Runtime > Change runtime type > A100 GPU, then run Cells 1, 2, 3, 7 and this cell again.")
    torch.manual_seed(0)
    FINISHER = make_unet().to(DEVICE)
    opt = torch.optim.AdamW(FINISHER.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=0.05)
    XT = torch.from_numpy(X_TRAIN)
    YT = torch.from_numpy(Y_TRAIN)
    base_wrong = 100.0 * float(((X_VAL >= 128) != Y_VAL).mean())
    print(f"Before training: {base_wrong:.2f}% of validation pixels differ from the designer")
    t0 = time.time()
    for step in range(1, STEPS + 1):
        idx = torch.randint(0, len(XT), (BATCH,))
        x = XT[idx].float().div(255).unsqueeze(1).to(DEVICE, non_blocking=True)
        y = YT[idx].float().unsqueeze(1).to(DEVICE, non_blocking=True)
        x, y = augment(x, y)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
            logits = run_unet(FINISHER, x)
        loss = F.binary_cross_entropy_with_logits(logits.float(), y, weight=edge_weights(x, y))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
        if step % max(1, STEPS // 10) == 0 or step == STEPS:
            w, fixed, broke = evaluate(FINISHER, X_VAL, Y_VAL)
            print(f"step {step:6d}/{STEPS}  loss {loss.item():.4f} | validation: {w:.2f}% pixels wrong (was {base_wrong:.2f}%),"
                  f" {fixed:.1f}% of needed fixes made, {broke:.2f}% of good pixels spoiled   [{time.time() - t0:.0f}s]")
    FINISHER.eval()
    with torch.no_grad():
        xv = torch.from_numpy(X_VAL[:1000]).float().div(255).unsqueeze(1).to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
            pv = (run_unet(FINISHER, xv).float() > 0).float().mean().item()
    print(f"Red on validation crops: input {100 * float((X_VAL[:1000] >= 128).mean()):.1f}%  ->  model {100 * pv:.1f}%"
          f"  (designer {100 * float(Y_VAL[:1000].mean()):.1f}%)")
    torch.save(FINISHER.state_dict(), MODEL_DIR / MODEL_NAME)
    print("Saved model      :", MODEL_DIR / MODEL_NAME)
else:
    print('Finisher training skipped.')


In [ ]:
if V5_USE_LEARNED_FINISHER:
    V5_CORE.require_input_context(globals())
    # ==================================================================================================
    # CELL 9: APPLY THE FINISHER  -  the trained model finishes the REAL Photoshop export of DESIGN (a design it
    # never saw in training) and is compared with Photoshop, Cell 6 and the designer. Saves BMP + PNG for review.
    # Needs Cells 1, 2, 3 and either Cell 8 in this session or a saved finisher_v1.pt on Drive.
    # ==================================================================================================
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    MODEL_FILE = PIPE / "05_Finisher_Model" / "finisher_v2.pt"     # v2: also handles pure red / white exports
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


    def block(cin, cout):
        return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                             nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


    def make_unet(ch=(32, 64, 128, 256, 384)):
        m = nn.ModuleDict()
        m["downs"] = nn.ModuleList([block(a, b) for a, b in zip((1,) + ch[:-1], ch)])
        m["upconv"] = nn.ModuleList([nn.ConvTranspose2d(a, b, 2, stride=2) for a, b in zip(ch[:0:-1], ch[-2::-1])])
        m["ups"] = nn.ModuleList([block(2 * b, b) for b in ch[-2::-1]])
        m["head"] = nn.Conv2d(ch[0], 1, 1)
        return m


    def run_unet(m, x):
        skips = []
        for i, d in enumerate(m["downs"]):
            x = d(x)
            if i < len(m["downs"]) - 1:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, blk in zip(m["upconv"], m["ups"]):
            x = up(x)
            x = blk(torch.cat([x, skips.pop()], 1))
        return m["head"](x)


    def finish_learned(model, soft):
        """Photoshop redness (H x W, 0..1) -> designer-style red / white (H x W bool)."""
        h, w = soft.shape
        ph, pw = (-h) % 16, (-w) % 16
        x = torch.from_numpy(np.pad(soft.astype(np.float32), ((0, ph), (0, pw)))).float()[None, None].to(DEVICE)
        model.eval()
        with torch.no_grad():
            p = torch.sigmoid(run_unet(model, x).float())[0, 0, :h, :w].cpu().numpy()
        return p >= 0.5, p


    def gap_rows(red):
        """Separation width as the designer counts it (white pixels per row / column across a line between two
        different pieces): % of line cross-sections that are 1 / 2 / 3 / 4 px."""
        lab = ndi.label(red, structure=N8)[0]
        Hh, Ww = red.shape
        hl = np.full((Hh, Ww), 99)
        vl = np.full((Hh, Ww), 99)
        for arr, L in ((lab, hl), (lab.T, vl.T)):
            for i in range(arr.shape[0]):
                row = arr[i]
                j = 0
                while j < len(row):
                    if row[j] == 0:
                        k = j
                        while k + 1 < len(row) and row[k + 1] == 0:
                            k += 1
                        if j > 0 and k + 1 < len(row) and row[j - 1] != row[k + 1] and k - j + 1 <= 6:
                            L[i, j:k + 1] = k - j + 1
                        j = k + 1
                    else:
                        j += 1
        wmin = np.minimum(hl, vl)
        cnt = {g: float(((wmin == g) / g).sum()) for g in (1, 2, 3, 4)}
        tot = max(1e-9, sum(cnt.values()))
        return {g: round(100 * v / tot, 1) for g, v in cnt.items()}


    # ---------------- run ----------------
    if "FINISHER" not in globals():
        FINISHER = make_unet().to(DEVICE)
        FINISHER.load_state_dict(torch.load(MODEL_FILE, map_location=DEVICE))
    RED_LEARN, PROB_LEARN = finish_learned(FINISHER, SOFT_PS)
    Image.fromarray(rgb_of(RED_LEARN)).save(DESIGN_DIR / "5 learned finisher.png")
    Image.fromarray(rgb_of(RED_LEARN)).save(DESIGN_DIR / "5 learned finisher.bmp")
    m_learn = measure(RED_LEARN, REF)

    cols = [("Photoshop", RED_PS)]
    cell6_file = DESIGN_DIR / "4 petal groups + curve rule.png"
    if "RED_FINAL" in globals():
        cols.append(("Cell 6", RED_FINAL))
    elif cell6_file.exists():
        cols.append(("Cell 6", redness_of(Image.open(cell6_file)) >= 0.5))
    cols.append(("learned", RED_LEARN))
    if REF is not None:
        cols.append(("designer", REF))
    meas = [measure(m, REF) for t, m in cols]
    gaps = [gap_rows(m) for t, m in cols]

    print("=" * 100)
    print(f"CELL 9: LEARNED FINISHER  -  design {DESIGN} (never seen in training)")
    print("=" * 100)
    print("Pixels changed   :", int((RED_LEARN != RED_PS).sum()), "of", W * H, "  | unsure pixels (40-60%):",
          int(((PROB_LEARN > 0.4) & (PROB_LEARN < 0.6)).sum()))
    print()
    print(f"{'measure':40s}" + "".join(f"{t:>11s}" for t, m in cols))
    for g, label in ((2, "lines 2 px per row/col % (target)"), (1, "lines 1 px per row/col %"), (3, "lines 3 px per row/col %"),
                     (4, "lines 4 px per row/col %")):
        print(f"{label:40s}" + "".join(f"{str(r[g]):>11s}" for r in gaps))
    for k in ("red %", "pieces", "A/M/B corner screening %", "bumps / nicks", "red specks (1-3 px)",
              "pin-holes / broken lines", "shape match with designer %"):
        print(f"{k:40s}" + "".join(f"{str(r.get(k, '')):>11s}" for r in meas))

    views = [(100, 80, 133, 106), (0, 0, W // 2, H // 2), (W // 2, H // 2, W - W // 2, H - H // 2)]
    for x0, y0, bw, bh in views:
        fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 6 * bh / bw + 0.5))
        for a, (t, m) in zip(ax, cols):
            a.imshow(rgb_of(m)[y0:y0 + bh, x0:x0 + bw], interpolation="nearest")
            a.set_title(f"{t}  x{x0}-{x0 + bw} y{y0}-{y0 + bh}")
            a.axis("off")
        plt.tight_layout()
        plt.show()
    V5_LEARNED_CONTEXT = dict(V5_ACTIVE_INPUT)
    V5_CORE.record_stage(globals(), "Cell 9: learned candidate (not automatically accepted)", RED_LEARN)

else:
    print('Saved learned finisher skipped: geometry-only mode.')


In [ ]:
V5_CORE.require_input_context(globals())
# Retained only for optional candidate proposals, not the default final path.
def separation_guard(red, ps, prob):
    """Cut pieces the model joined along a 2-px line between Photoshop's pieces; fill stray pin-holes."""
    Hh, Ww = red.shape
    lab_ps, nps = ndi.label(ps, structure=N8)
    dist, (iy, ix) = ndi.distance_transform_edt(~ps, return_indices=True)
    near = lab_ps[iy, ix]                                   # every pixel -> nearest Photoshop piece
    p4 = np.pad(near, 1)
    line = np.zeros_like(red)
    for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        line |= p4[1 + dy:Hh + 1 + dy, 1 + dx:Ww + 1 + dx] != near
    out = red.copy()
    lab_m, nm = ndi.label(red, structure=N8)
    cut = 0
    for k, sl in enumerate(ndi.find_objects(lab_m), start=1):
        comp = lab_m[sl] == k
        inside = np.unique(lab_ps[sl][comp & ps[sl]])
        inside = inside[inside > 0]
        if inside.size > 1:                                 # one model piece covers several Photoshop pieces
            sub = out[sl]
            sub[comp & line[sl] & np.isin(near[sl], inside)] = False
            cut += 1
    for rnd in range(3):                                    # pieces may not touch at a pixel corner
        own = np.where(out, near, 0)
        changed = False
        for a, b, oa, ob in ((own[:-1, :-1], own[1:, 1:], (0, 0), (1, 1)), (own[:-1, 1:], own[1:, :-1], (0, 1), (1, 0))):
            for y, x in np.argwhere((a > 0) & (b > 0) & (a != b)):
                p1, p2 = (y + oa[0], x + oa[1]), (y + ob[0], x + ob[1])
                out[p1 if prob[p1] <= prob[p2] else p2] = False
                changed = True
        if not changed:
            break
    wl, wn = ndi.label(~out, structure=N4)                  # stray pin-holes inside one piece
    size = np.bincount(wl.ravel())
    size[0] = 99
    lab_o = ndi.label(out, structure=N8)[0]
    filled = 0
    for k in np.nonzero(size <= 2)[0]:
        sp = wl == k
        ring = ndi.binary_dilation(sp, N8) & out
        if np.unique(lab_o[ring]).size == 1 and not (sp & line).any():
            out[sp] = True
            filled += 1
    return out, {"pieces re-cut": cut, "pin-holes filled": filled}

# CELL 10: FINAL — conservative pass from the ORIGINAL Photoshop draft.
# Independent of intermediate RED_FINAL/RED_SEP; never uses designer truth to select edits.
V5_CORE.require_input_context(globals())
V5_CORE.install_bindings(globals())

proposal = None
if V5_USE_LEARNED_FINISHER:
    if globals().get('V5_LEARNED_CONTEXT') != V5_ACTIVE_INPUT or 'RED_LEARN' not in globals():
        raise ValueError('Run Cell 9 for this design before enabling its learned proposal.')
    proposal = RED_LEARN

RED_DONE, V5_LAST_FINISHING_REPORT = V5_CORE.finish_from_draft(
    RED_PS, learned_proposal=proposal,
    max_shift=globals().get('MAX_SHIFT', 2),
    max_repeat=globals().get('MAX_REPEAT', 8),
    max_edits=globals().get('V5_MAX_CORNER_EDITS', 250),
    max_changed_fraction=V5_MAX_CHANGED_FRACTION,
    protected_white=globals().get('V5_PROTECTED_WHITE'))
V5_CORE.record_stage(globals(), 'FINAL: Photoshop plus bounded staircase edits', RED_DONE)
bmp, png, pure_ok = save_production(RED_DONE, DESIGN)
Image.fromarray(rgb_of(RED_DONE)).save(DESIGN_DIR / '6 final conservative staircase.png')

# Separate diagnostic image: blue = red removed, orange = red added.
# These colours are never saved in the Step_3 output.
difference = np.full(RED_PS.shape + (3,), 255, np.uint8)
difference[RED_PS & ~RED_DONE] = (0, 100, 255)
difference[RED_DONE & ~RED_PS] = (255, 160, 0)
Image.fromarray(difference).save(DESIGN_DIR / '6 changed pixels only.png')
(DESIGN_DIR / '6 stage changes.json').write_text(json.dumps(V5_STAGE_REPORTS, indent=2))

print('=' * 90)
print('CELL 10: CONSERVATIVE FINAL — design', DESIGN)
print('Source: original Photoshop draft from Cell 3.')
print('Broad gap/pocket-fill previews are not selected as the final baseline.')
print('Saved:', bmp)
print('      ', png)
print('Native palette/dimensions:', 'OK' if pure_ok else 'CHECK')
print('Changed pixels:', V5_LAST_FINISHING_REPORT['changed_pixels'], '/', W * H)
print('Red added:', V5_LAST_FINISHING_REPORT['red_added'],
      '| red removed:', V5_LAST_FINISHING_REPORT['red_removed'])
print('Automatic edit budget:', f'{100 * V5_MAX_CHANGED_FRACTION:.2f}%',
      '(this is not a manual-work estimate)')
print('Learned proposal:', V5_LAST_FINISHING_REPORT['learned_proposal'])
print('Detected corners:', V5_LAST_FINISHING_REPORT['audit'])
if V5_LAST_FINISHING_REPORT['changed_pixels'] == 0:
    print('No feasible verified edits accepted: original Photoshop pixels retained for review.')
print('Status:', V5_LAST_EXPORT_REPORT['status'])
print('Stage-by-stage changes relative to Photoshop:')
for row in V5_STAGE_REPORTS.values():
    print(' ', row['stage'], ':', row['changed_pixels_vs_photoshop'], 'pixels changed,',
          row['red_added'], 'red added,', row['red_removed'], 'red removed')

# Designer reference is comparison-only. These scores NEVER choose the output.
if REF is not None:
    comparisons = {'Photoshop': V5_CORE.native_similarity(REF, RED_PS),
                   'Conservative final': V5_CORE.native_similarity(REF, RED_DONE)}
    for label, values in comparisons.items():
        print(label, '— exact agreement:', f"{values['pixel_agreement_pct']:.2f}%",
              '| red IoU:', f"{values['red_iou_pct']:.2f}%",
              '| different pixels:', values['different_pixels'])
    (DESIGN_DIR / '6 native comparisons.json').write_text(json.dumps(comparisons, indent=2))

views = [('Photoshop', RED_PS), ('Conservative FINAL', RED_DONE)]
if REF is not None:
    views.append(('designer — comparison only', REF))
for box in [(0, 0, W, H), (100, 80, 133, 106)]:
    x0 = min(max(0, box[0]), W - 1)
    y0 = min(max(0, box[1]), H - 1)
    bw, bh = min(box[2], W - x0), min(box[3], H - y0)
    fig, axes = plt.subplots(1, len(views), figsize=(6 * len(views), max(2, 6 * bh / bw)), squeeze=False)
    for ax, (label, image) in zip(axes.ravel(), views):
        ax.imshow(rgb_of(image)[y0:y0+bh, x0:x0+bw], interpolation='nearest')
        ax.set_title(f'{label} x{x0}-{x0+bw} y{y0}-{y0+bh}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
print('The result is a REVIEW DRAFT. Unresolved contours and semantic details still need designer review.')


In [ ]:
if V5_RUN_LLM_REVIEW and V5_USE_LEARNED_FINISHER:
    V5_CORE.require_input_context(globals())
    # ==================================================================================================
    # CELL 11: LLM REVIEW + FIX  -  the reasoning model reviews the Cell 10 result tile by tile (next to the Step 2
    # sketch) and flags spots that break the business rules. For every flagged spot the code makes alternatives
    # (the finisher seen from 8 views, a stricter and a looser finisher, the rule-only Cell 6, Photoshop's own
    # drawing); the model picks the best one; the code applies it only if it joins no pieces and adds no bumps.
    # Saves the reviewed BMP + PNG (as v2) and a review sheet for the designer.
    # Needs Cells 2, 3, 4, 5, 9 and 10 in this session, and the OPENAI_API_KEY secret (Colab: key icon on the left).
    # ==================================================================================================
    import io
    import sys
    import base64
    import subprocess
    from concurrent.futures import ThreadPoolExecutor
    from PIL import ImageDraw
    try:
        from openai import OpenAI
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
        from openai import OpenAI
    from google.colab import userdata

    REASONING_MODEL = "gpt-6-astra"     # the reasoning model (vision) used for review and judging
    EFFORT = "medium"                   # reasoning effort: low / medium / high
    TILE = 80                           # review tiles are TILE x TILE pixels of the design
    MAX_SPOTS = 14                      # at most this many spots are judged per design
    CLIENT = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

    RULES = """GNR saree production design rules (red / white pixel artwork, every pixel counts):
    Rule A: every separate component (leaf, petal, bud, flower, stem, dot) is separated from its neighbours by a continuous white line exactly 2 pixels wide; components of one motif must not be joined.
    Rule B: the shapes follow the Step 2 sketch (same motifs, positions, proportions); nothing missing, nothing invented.
    Rule C: petals / leaves of one group are packed together and meet along thin white lines, so the group reads as one clean shape (not scattered blobs with wide white gaps).
    Rule D: an applicable curved corner has flat anchors A/B and a shared 1-pixel boundary transition M. From either flat toward M, run lengths start at 2/3/4, hold or decrease one level at a time to 1. Example horizontal 3,2,2 -> M(1) -> vertical 2,2,3. Flat lengths are free; there is no step-count dominance. Preserve shape in all orientations.
    Rule E: no stray pixels, 1-pixel bumps, notches, specks or broken lines; dots are round; repeated shapes are consistent."""

    ISSUES = ["joined pieces (separation missing)", "separation too wide or uneven", "scattered petal group",
              "jagged or wobbly curve", "stray pixels / bump / broken line", "shape does not follow the sketch",
              "dot not round", "other"]
    REVIEW_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["spots"],
                     "properties": {"spots": {"type": "array", "items": {
                         "type": "object", "additionalProperties": False,
                         "required": ["tile", "x0", "y0", "x1", "y1", "issue", "note"],
                         "properties": {"tile": {"type": "integer"}, "x0": {"type": "integer"}, "y0": {"type": "integer"},
                                        "x1": {"type": "integer"}, "y1": {"type": "integer"},
                                        "issue": {"type": "string", "enum": ISSUES}, "note": {"type": "string"}}}}}}
    JUDGE_SCHEMA = {"type": "object", "additionalProperties": False, "required": ["choice", "confidence", "reason"],
                    "properties": {"choice": {"type": "string", "enum": list("ABCDEF")},
                                   "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                                   "reason": {"type": "string"}}}


    def png_url(img):
        buf = io.BytesIO()
        img.save(buf, "PNG")
        return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


    def ask(text, images, schema, name):
        content = [{"type": "input_text", "text": text}] + [{"type": "input_image", "image_url": png_url(im), "detail": "high"}
                                                            for im in images]
        r = CLIENT.responses.create(model=REASONING_MODEL, reasoning={"effort": EFFORT},
                                    input=[{"role": "user", "content": content}],
                                    text={"format": {"type": "json_schema", "name": name, "schema": schema, "strict": True}})
        return json.loads(r.output_text)


    def panel(arr_rgb, zoom, grid=True, box=None):
        """Zoomed crop with a light grid every 10 px (so positions can be read) and an optional blue box."""
        h, w = arr_rgb.shape[:2]
        im = Image.fromarray(arr_rgb).resize((w * zoom, h * zoom), Image.NEAREST)
        d = ImageDraw.Draw(im)
        if grid:
            for v in range(10, max(w, h), 10):
                if v < w:
                    d.line([(v * zoom, 0), (v * zoom, h * zoom)], fill=(150, 190, 255))
                if v < h:
                    d.line([(0, v * zoom), (w * zoom, v * zoom)], fill=(150, 190, 255))
        if box is not None:
            x0, y0, x1, y1 = box
            d.rectangle([x0 * zoom, y0 * zoom, x1 * zoom - 1, y1 * zoom - 1], outline=(0, 90, 255), width=3)
        return im


    SKETCH_RGB = np.asarray(sketch.convert("RGB"))


    def tile_image(k, tx, ty, red):
        """Tile k: Step 2 sketch | current result, zoomed 4x, with grid and a header."""
        z = 4
        sk = SKETCH_RGB[ty:ty + TILE, tx:tx + TILE]
        rs = rgb_of(red[ty:ty + TILE, tx:tx + TILE])
        a, b = panel(sk, z), panel(rs, z)
        t = Image.new("RGB", (a.width + b.width + 12, a.height + 22), (255, 255, 255))
        ImageDraw.Draw(t).text((4, 4), f"TILE {k}  (design x{tx}-{tx + sk.shape[1]}, y{ty}-{ty + sk.shape[0]})   left: Step 2 sketch   right: result", fill=(0, 0, 0))
        t.paste(a, (0, 22))
        t.paste(b, (a.width + 12, 22))
        return t


    def finish_variant(prob, thr):
        """The Cell 10 pipeline on a finisher probability map with a given red threshold."""
        r, g = separation_guard(prob >= thr, RED_PS, prob)
        r = tidy(r, np.ones_like(r))
        r, n1 = fix_curves(r)
        r, n2 = fix_curves(r)
        return tidy(r, np.ones_like(r))


    def tta_prob(soft):
        """Finisher probability averaged over the 8 rotations / mirrors of the design."""
        acc = np.zeros(soft.shape, np.float32)
        for k in range(4):
            for flip in (False, True):
                s = np.rot90(soft, k)
                s = s[:, ::-1] if flip else s
                pr = finish_learned(FINISHER, np.ascontiguousarray(s))[1]
                pr = pr[:, ::-1] if flip else pr
                acc += np.rot90(pr, -k)
        return acc / 8.0


    # ---------------- 1. review ----------------
    t0 = datetime.now()
    tiles = [(tx, ty) for ty in range(0, H, TILE) for tx in range(0, W, TILE)]
    review_text = (RULES + "\n\nYou are a senior GNR textile designer checking a production design before it goes to the loom. "
                   "Each image shows numbered tiles: the Step 2 pencil sketch on the left and the red / white result on the "
                   "right, zoomed 4x with a light blue grid line every 10 design pixels. Flag ONLY clear problems a designer "
                   "would fix. For each problem give the tile number and a box in that tile's own pixel coordinates "
                   f"(0 to {TILE}; x to the right, y down, read from the grid), the issue type and a short note. "
                   "Return an empty list if the tiles are fine.")
    batches = [list(range(i, min(i + 4, len(tiles)))) for i in range(0, len(tiles), 4)]


    def review_batch(ids):
        imgs = [tile_image(k + 1, tiles[k][0], tiles[k][1], RED_DONE) for k in ids]
        return ask(review_text + f"\nTiles in this message: {[k + 1 for k in ids]}.", imgs, REVIEW_SCHEMA, "review")["spots"]


    with ThreadPoolExecutor(6) as ex:
        found = [s for res in ex.map(review_batch, batches) for s in res]
    spots = []
    for s in found:
        k = s["tile"] - 1
        if not 0 <= k < len(tiles):
            continue
        tx, ty = tiles[k]
        bx0, bx1 = sorted((min(max(s["x0"], 0), TILE), min(max(s["x1"], 0), TILE)))
        by0, by1 = sorted((min(max(s["y0"], 0), TILE), min(max(s["y1"], 0), TILE)))
        bx1, by1 = max(bx1, bx0 + 8), max(by1, by0 + 8)          # at least an 8 x 8 px spot
        box = (max(0, tx + bx0 - 3), max(0, ty + by0 - 3), min(W, tx + bx1 + 3), min(H, ty + by1 + 3))
        spots.append(dict(s, box=box))
    spots = spots[:MAX_SPOTS]

    # ---------------- 2. alternatives ----------------
    PROB_TTA = tta_prob(SOFT_PS)
    options = [("current result (Cell 10)", RED_DONE), ("finisher, 8-view average", finish_variant(PROB_TTA, 0.5)),
               ("finisher, less red", finish_variant(PROB_TTA, 0.6)), ("finisher, more red", finish_variant(PROB_TTA, 0.4))]
    if "RED_FINAL" in globals():
        try:
            V5_CORE.require_stage(globals(), "Cell 6: pocket fill preview", RED_FINAL)
        except ValueError:
            print("Skipping stale/unavailable Cell 6 candidate.")
        else:
            options.append(("rules only (Cell 6)", RED_FINAL))
    options.append(("Photoshop original", RED_PS))
    LETTERS = "ABCDEF"[:len(options)]

    # ---------------- 3. judge ----------------
    judge_text = (RULES + "\n\nYou are a senior GNR textile designer. The first image is the Step 2 sketch of one area; the "
                  "next images are candidate versions of the SAME area, labelled A, B, C ... (A = the current result). The blue "
                  "box marks the spot that was flagged: '{issue}' - {note}. Choose the candidate that best follows the rules and "
                  "the sketch INSIDE the blue box, without harming the rest. If none is better than A, choose A. "
                  "Candidates: {names}.")


    def judge(s):
        x0, y0, x1, y1 = s["box"]
        m = 10
        cx0, cy0, cx1, cy1 = max(0, x0 - m), max(0, y0 - m), min(W, x1 + m), min(H, y1 + m)
        box = (x0 - cx0, y0 - cy0, x1 - cx0, y1 - cy0)
        present = [i for i, (n, o) in enumerate(options) if i == 0 or (o[y0:y1, x0:x1] != RED_DONE[y0:y1, x0:x1]).any()]
        if len(present) == 1:
            return dict(s, choice="A", confidence="high", reason="no alternative differs here", present=present)
        imgs = [panel(SKETCH_RGB[cy0:cy1, cx0:cx1], 6, grid=False, box=box)]
        names = []
        for j, i in enumerate(present):
            im = panel(rgb_of(options[i][1][cy0:cy1, cx0:cx1]), 6, grid=False, box=box)
            lab = Image.new("RGB", (im.width, im.height + 26), (255, 255, 255))
            ImageDraw.Draw(lab).text((4, 6), f"candidate {LETTERS[j]}", fill=(0, 0, 0))
            lab.paste(im, (0, 26))
            imgs.append(lab)
            names.append(f"{LETTERS[j]} = {options[i][0]}")
        ans = ask(judge_text.format(issue=s["issue"], note=s["note"], names="; ".join(names)), imgs, JUDGE_SCHEMA, "judge")
        letter = ans["choice"] if ans["choice"] in LETTERS[:len(present)] else "A"
        return dict(s, choice=letter, confidence=ans["confidence"], reason=ans["reason"], present=present)


    with ThreadPoolExecutor(6) as ex:
        verdicts = list(ex.map(judge, spots))

    # ---------------- 4. apply (only if no pieces join and no bumps are added) ----------------
    RED_REVIEWED = RED_DONE.copy()
    for v in verdicts:
        v["applied"] = False
        if v["choice"] == "A" or v["confidence"] == "low":
            continue
        opt = options[v["present"][LETTERS.index(v["choice"])]][1]
        x0, y0, x1, y1 = v["box"]
        trial = RED_REVIEWED.copy()
        trial[y0:y1, x0:x1] = opt[y0:y1, x0:x1]
        seam = np.zeros_like(trial)
        seam[max(0, y0 - 2):y1 + 2, max(0, x0 - 2):x1 + 2] = True
        seam[y0 + 2:y1 - 2, x0 + 2:x1 - 2] = False
        trial = tidy(trial, seam)
        win = (slice(max(0, y0 - 6), y1 + 6), slice(max(0, x0 - 6), x1 + 6))
        joins = ndi.label(trial[win], structure=N8)[1] < ndi.label(RED_REVIEWED[win], structure=N8)[1]
        more_bumps = bump_count(trial[win]) > bump_count(RED_REVIEWED[win])
        checked_trial, geometry_report = V5_CORE.consider_proposal(RED_PS, RED_REVIEWED, trial,
            max_changed_fraction=V5_MAX_CHANGED_FRACTION, protected_white=V5_PROTECTED_WHITE)
        geometry_ok = geometry_report["accepted"]
        v["geometry_check"] = {"passed": geometry_ok, "reason": geometry_report["reason"]}
        if not joins and not more_bumps and geometry_ok:
            RED_REVIEWED = checked_trial
            v["applied"] = True
        else:
            v["reason"] += "  [not applied: " + ("would join pieces" if joins else "would add bumps" if more_bumps else geometry_report["reason"]) + "]"

    V5_CORE.record_stage(globals(), "Cell 11: guarded LLM result", RED_REVIEWED)
    bmp2, png2, pure2 = save_production(RED_REVIEWED, DESIGN)

    # ---------------- 5. review sheet for the designer ----------------
    rows = []
    for n, v in enumerate(verdicts, start=1):
        x0, y0, x1, y1 = v["box"]
        m = 10
        cx0, cy0, cx1, cy1 = max(0, x0 - m), max(0, y0 - m), min(W, x1 + m), min(H, y1 + m)
        box = (x0 - cx0, y0 - cy0, x1 - cx0, y1 - cy0)
        ims = [panel(SKETCH_RGB[cy0:cy1, cx0:cx1], 5, False, box), panel(rgb_of(RED_DONE[cy0:cy1, cx0:cx1]), 5, False, box),
               panel(rgb_of(RED_REVIEWED[cy0:cy1, cx0:cx1]), 5, False, box)]
        status = ("FIXED with " + options[v["present"][LETTERS.index(v["choice"])]][0]) if v["applied"] else "FOR THE DESIGNER"
        head = f"#{n}  x{x0}-{x1} y{y0}-{y1}  {v['issue']}  ->  {status} ({v['confidence']})"
        tw = sum(i.width for i in ims) + 24
        t = Image.new("RGB", (max(tw, 900), max(i.height for i in ims) + 58), (255, 255, 255))
        d = ImageDraw.Draw(t)
        d.text((4, 4), head, fill=(0, 0, 0) if v["applied"] else (200, 0, 0))
        d.text((4, 20), ("note: " + v["note"] + "  |  why: " + v["reason"])[:160], fill=(70, 70, 70))
        xx = 0
        for i in ims:
            t.paste(i, (xx, 40))
            xx += i.width + 12
        rows.append(t)
    if rows:
        sheet_img = Image.new("RGB", (max(r.width for r in rows), sum(r.height + 10 for r in rows)), (235, 235, 235))
        yy = 0
        for r in rows:
            sheet_img.paste(r, (0, yy))
            yy += r.height + 10
        sheet_img.save(DESIGN_DIR / "7 LLM review sheet.png")
    log = [{k: (list(map(int, v[k])) if k == "box" else v[k]) for k in ("tile", "box", "issue", "note", "choice", "confidence",
                                                                          "reason", "applied")} for v in verdicts]
    (DESIGN_DIR / "7 LLM review.json").write_text(json.dumps(log, indent=2), encoding="utf-8")

    # ---------------- report ----------------
    print("=" * 100)
    print(f"CELL 11: LLM REVIEW + FIX  -  design {DESIGN}   ({str(datetime.now() - t0).split('.')[0]})")
    print("=" * 100)
    print("Spots flagged    :", len(found), "(judged:", len(verdicts), ")   fixed:", sum(v["applied"] for v in verdicts),
          "  left for the designer:", sum(not v["applied"] for v in verdicts))
    print("Saved            :", bmp2)
    print("                   ", png2, "  (pure red / white, exact size:", "OK" if pure2 else "CHECK", ")")
    print("Review sheet     :", DESIGN_DIR / "7 LLM review sheet.png")
    print()
    for n, v in enumerate(verdicts, start=1):
        status = "FIXED " if v["applied"] else "DESIGNER"
        print(f"#{n:2d} {status:8s} tile {v['tile']:2d} box {v['box']}  {v['issue']:34s} -> {v['choice']} ({v['confidence']}): {v['reason'][:90]}")
    cols = [("Photoshop", RED_PS), ("Cell 10", RED_DONE), ("reviewed", RED_REVIEWED)] + ([("designer", REF)] if REF is not None else [])
    meas = [measure(m, REF) for t, m in cols]
    gaps = [gap_rows(m) for t, m in cols]
    print()
    print(f"{'measure':40s}" + "".join(f"{t:>11s}" for t, m in cols))
    print(f"{'lines 2 px per row/col % (target)':40s}" + "".join(f"{str(r[2]):>11s}" for r in gaps))
    for k in ("red %", "pieces", "A/M/B corner screening %", "bumps / nicks", "pin-holes / broken lines", "shape match with designer %"):
        print(f"{k:40s}" + "".join(f"{str(r.get(k, '')):>11s}" for r in meas))
    if REF is not None:
        print(f"{'pixels different from designer':40s}" + "".join(f"{int((m != REF).sum()):>11d}" for t, m in cols))
    if rows:
        plt.figure(figsize=(16, 16 * sheet_img.height / sheet_img.width))
        plt.imshow(sheet_img)
        plt.axis("off")
        plt.title("LLM review: sketch | Cell 10 | reviewed  (blue box = flagged spot)")
        plt.show()
else:
    print('Paid LLM review skipped.')


In [ ]:
if V5_RUN_LEGACY_BATCH:
    # ==================================================================================================
    # CELL 12: BATCH  -  finishes every design in BATCH that has a Photoshop export in the inbox, in one run:
    #   import -> learned finisher -> separation guard -> tidy -> curve rule -> production BMP + PNG
    # A copy for the designers goes into 06_Edit_Pairs/<BATCH_NAME>; they finish it in Paint Shop Pro and save
    # their own version next to it (pipeline output + designer edit = a training pair).
    # Runs on a fresh runtime on its own (the functions of Cells 1, 3, 4, 5, 9, 10 are loaded from
    # Notebook/gnr_v4_lib.py). Designs already done are skipped: when more exports arrive, just run it again.
    # ==================================================================================================
    BATCH = ["828", "261", "827", "694", "371", "752", "1433", "95", "1191", "794"]
    BATCH_NAME = "Batch_01"
    REDO = []                       # design numbers to finish again (written as a new version, nothing is overwritten)

    import time
    import shutil
    from pathlib import Path
    from google.colab import drive

    drive.mount("/content/drive")
    DATASET = Path("/content/drive/MyDrive/GNR Design Data-Set")
    STAGE2 = DATASET / "Stage_2"                       # read only
    STAGE3 = DATASET / "Stage_3"                       # read only (designer's file, used only to compare)
    PIPE = DATASET / "GNR_Step3_Photoshop_Pipeline"
    INBOX = PIPE / "01_Photoshop_Inbox"
    OUTPUT = PIPE / "02_Production_Output"
    REVIEW = PIPE / "03_Review"
    HAND_OUT = PIPE / "06_Edit_Pairs" / BATCH_NAME
    for folder in (INBOX, OUTPUT, REVIEW, HAND_OUT):
        folder.mkdir(parents=True, exist_ok=True)
    exec(V5_COMPAT_PATH.read_text(encoding="utf-8"))

    S2 = pair_files(STAGE2, 2)
    S3 = pair_files(STAGE3, 3)
    if V5_USE_LEARNED_FINISHER and "FINISHER" not in globals():
        FINISHER = make_unet().to(DEVICE)
        FINISHER.load_state_dict(torch.load(MODEL_FILE, map_location=DEVICE))


    def find_export(d):
        """Newest image in the inbox whose name starts with the design number and that is a red drawing
        (a Step 2 sketch is skipped, whatever its name). An export saved into 00_Photoshop_Input by mistake
        is found too."""
        found = None
        files = list(INBOX.iterdir()) + list((PIPE / "00_Photoshop_Input").glob("*"))
        for p in sorted(files, key=lambda q: q.stat().st_mtime):
            words = p.stem.replace(chr(95), " ").split()
            if not (p.is_file() and p.suffix.lower() in (".png", ".bmp", ".jpg", ".jpeg") and words and words[0] == d):
                continue
            a = np.asarray(Image.open(p).convert("RGB")).astype(np.float32) / 255.0
            if float(((a[..., 0] - (a[..., 1] + a[..., 2]) / 2) > 0.3).mean()) >= 0.05:
                found = p
        return found


    def next_free(path):
        i = 2
        out = path
        while out.exists():
            out = path.with_name(f"{path.stem}{chr(95)}v{i}{path.suffix}")
            i += 1
        return out


    README = f"""GNR Step 3 - designer finishing, {BATCH_NAME}

    For every design in this folder there are two files:
       <design> GNR 2 - sketch.png       the Step 2 sketch (for reference only)
       <design> GNR 3 - pipeline.bmp     the AI pipeline's Step 3 (pure red / white, exact Step 2 size)

    Please, for each design:
    Step A: open "<design> GNR 3 - pipeline.bmp" in Paint Shop Pro and finish it the way you normally would
            (2-px separations, curves, dots, everything). Do not change the image size.
    Step B: save it as BMP into THIS folder, named  "<design> GNR 3 - <your name>.bmp"
            for example  "828 GNR 3 - Ravi.bmp".  Never overwrite the pipeline file.
    Step C: write down how many minutes the finishing took (tell NagaRaju, or add it to the name:
            "828 GNR 3 - Ravi - 25 min.bmp").
    Do NOT add the company branding - only the design finishing.
    Your finished files teach the AI how you finish, so the next version needs less work.
    """
    (HAND_OUT / "README - for designers.txt").write_text(README, encoding="utf-8")

    rows, waiting, done_before, shown = [], [], [], []
    for d in dict.fromkeys(BATCH):
        if d not in S2:
            print("design", d, ": no Step 2 sketch in Stage_2 - skipped")
            continue
        hand_bmp = HAND_OUT / f"{d} GNR 3 - pipeline.bmp"
        if hand_bmp.exists() and d not in REDO:
            done_before.append(d)
            continue
        export = find_export(d)
        if export is None:
            waiting.append(d)
            continue
        t0 = time.time()
        sketch = Image.open(S2[d]).convert("RGB")
        W, H = sketch.size
        red_ps, soft, info = import_export(export, (W, H))
        learned_candidate = None
        if V5_USE_LEARNED_FINISHER:
            learned_candidate, prob = finish_learned(FINISHER, soft)
        RED_PS = red_ps.copy()
        V5_ACTIVE_INPUT = None  # Reload interactive Cell 3 after switching to batch images.
        V5_STAGE_REPORTS = {}
        red, V5_LAST_FINISHING_REPORT = _core.finish_from_draft(red_ps, learned_proposal=learned_candidate,
            max_changed_fraction=V5_MAX_CHANGED_FRACTION, max_shift=MAX_SHIFT, max_repeat=MAX_REPEAT,
            max_edits=V5_MAX_CORNER_EDITS, protected_white=V5_PROTECTED_WHITE)
        n1 = V5_LAST_FINISHING_REPORT['accepted_corner_edits']
        n2 = 0
        guard_info = V5_LAST_FINISHING_REPORT['learned_proposal']
        bmp, png, pure_ok = save_production(red, d)

        ddir = REVIEW / d
        ddir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(rgb_of(red_ps)).save(ddir / "1 photoshop pure.png")
        np.save(ddir / "photoshop_soft.npy", soft)
        Image.fromarray(rgb_of(red)).save(ddir / "6 final (learned + rules).png")
        hand_bmp = next_free(hand_bmp)
        shutil.copyfile(bmp, hand_bmp)
        sketch.save(HAND_OUT / f"{d} GNR 2 - sketch.png")

        ref = None
        if d in S3:
            ref_img = Image.open(S3[d])
            ref = redness_of(ref_img.resize((W, H), Image.NEAREST) if ref_img.size != (W, H) else ref_img) >= 0.5
        m_ps, m_fin = measure(red_ps), measure(red, ref)
        m_ref = measure(ref) if ref is not None else {}
        rows.append({"design": d, "size": f"{W}x{H}", "export": info["note"], "ok": "OK" if pure_ok else "CHECK",
                     "pieces": f"{m_ps['pieces']}>{m_fin['pieces']} ({m_ref.get('pieces', '-')})",
                     "red %": f"{m_fin['red %']} ({m_ref.get('red %', '-')})",
                     "curve rule %": f"{m_ps['A/M/B corner screening %']}>{m_fin['A/M/B corner screening %']} ({m_ref.get('A/M/B corner screening %', '-')})",
                     "bumps": f"{m_ps['bumps / nicks']}>{m_fin['bumps / nicks']} ({m_ref.get('bumps / nicks', '-')})",
                     "match %": m_fin.get("shape match with designer %", "-"), "file": bmp.name, "sec": round(time.time() - t0)})
        shown.append((d, sketch, red_ps, red))
        print(f"design {d:>5s} finished in {rows[-1]['sec']} s: {export.parent.name}/{export.name} -> {bmp.name}"
              f"  | guard {guard_info}  | curves rebuilt {n1 + n2}")

    print()
    print("=" * 120)
    print(f"CELL 12: BATCH {BATCH_NAME}  -  {len(rows)} finished now, {len(done_before)} done before, {len(waiting)} waiting for Photoshop")
    print("=" * 120)
    if rows:
        print("Photoshop > FINAL (designer's own file, comparison only)")
        cols = ["design", "size", "pieces", "red %", "curve rule %", "bumps", "match %", "ok", "sec", "export"]
        widths = [7, 9, 16, 13, 17, 12, 8, 6, 5, 30]
        print("".join(f"{c:<{w}s}" for c, w in zip(cols, widths)))
        for r in rows:
            print("".join(f"{str(r[c]):<{w}s}" for c, w in zip(cols, widths)))
    if done_before:
        print("Already done before (skipped):", ", ".join(done_before), "  - put a number in REDO to finish it again")
    if waiting:
        print("Waiting for the Photoshop export of:", ", ".join(waiting))
        print("   save each one into 01_Photoshop_Inbox as  '<design> GNR 3.png', then run this cell again")
    print()
    print("Production BMP + PNG  :", OUTPUT)
    print("For the designers     :", HAND_OUT, " (", len(list(HAND_OUT.glob("* - pipeline*.bmp"))), "pipeline BMPs + README )")

    for d, sketch, red_ps, red in shown:
        W, H = sketch.size
        fig, ax = plt.subplots(1, 3, figsize=(18, 6 * H / W + 0.6))
        for a, (t, im) in zip(ax, (("Step 2 sketch", sketch), ("Photoshop export", rgb_of(red_ps)), ("FINAL (for the designers)", rgb_of(red)))):
            a.imshow(im, interpolation="nearest")
            a.set_title(f"{d}: {t}")
            a.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print('Optional learned batch skipped; enable only with a saved model.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14: SUNBURST BATCH  -  Sunburst (gpt-image-2.5-sunburst) draws a Step 3 for each Step 2 sketch in DESIGNS.
    # Sunburst gets ONLY the Step 2 sketch (scaled onto its canvas) + the rules prompt, never the designer's Step 3.
    # Raw images go to GNR_Step3_Photoshop_Pipeline/07_Sunburst_Raw/  ("<design> GNR 3 sunburst.png" + prompt + canvas json).
    # The finishing (import to exact size, finisher, repeats, curve rule, petals, separations) runs afterwards on these files.
    # Self-contained: needs only the OPENAI_API_KEY secret (key icon on the left) and Google Drive.
    # ==================================================================================================
    DESIGNS = ["145", "166"]
    VERSION = " v2"                 # added to the file names, so earlier Sunburst images are kept ("" = first version)
    FORCE_REGENERATE = False        # True = call Sunburst again even if a raw image already exists (costs again)
    LONG_SIDE = 1440                # Sunburst canvas: the sketch is scaled so its long side is about this many px
    IMAGE_MODEL = "gpt-image-2.5-sunburst"
    IMAGE_QUALITY = "high"

    import base64
    import io
    import json
    import time
    from pathlib import Path
    import numpy as np
    from PIL import Image
    from google.colab import drive, userdata
    from openai import OpenAI, BadRequestError

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DATASET = Path("/content/drive/MyDrive/GNR Design Data-Set")
    STAGE2 = DATASET / "Stage_2"                                        # read only
    RAW = DATASET / "GNR_Step3_Photoshop_Pipeline" / "07_Sunburst_Raw"
    RAW.mkdir(parents=True, exist_ok=True)
    client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

    S2 = {}
    for p in STAGE2.iterdir():
        d = p.name.split()[0]
        if p.is_file() and d in DESIGNS and " GNR 2" in p.name:
            S2[d] = p


    def canvas_for(w, h):
        s = LONG_SIDE / max(w, h)
        cw, ch = (max(16, int(round(v * s / 16)) * 16) for v in (w, h))
        return cw, ch, s


    def prompt_for(w, h, cw, ch, s):
        two = max(2, int(round(2 * s)))
        return f"""Act as an expert textile designer and production finisher. Convert the supplied Step 2 sketch into a professional Step 3 red-and-white design, regardless of motif type.

    Analyze all design elements, their relationships, repeated families, curves, fills, and intentional white spaces.

    Apply these rules:
    Rule 1: Preserve the overall composition, every motif, its position, size, orientation and relationships exactly as in the sketch. Do not add, remove or move elements.
    Rule 2: The background is WHITE, even where the sketch shows it dark or shaded. Every motif is RED: motifs drawn solid in the sketch are solid red, with their inner details as white cut-outs; motifs drawn only as outlines or line work in the sketch stay line drawings (red lines on white), never filled solid. Avoid accidental holes.
    Rule 3: Keep independent components separated by white space. Every leaf, petal, bud and flower is separated from its stem and its neighbours by a continuous thin white line of even width (about {two} pixels on this canvas, 2 pixels at final size), the same width everywhere. Preserve deliberate connections and overlaps.
    Rule 4: Shapes are full and bold with even stroke thickness. Keep thin stems and dots as they are: do not merge leaves into stems or turn dots into other shapes.
    Rule 5: Curves are smooth with gradual tapering and clean pointed or rounded ends. Repeated component families keep the same shape, proportions and filling. All dots are perfectly round.
    Rule 6: Eliminate unintended hooks, blobs, stray pixels, broken strokes, protrusions, accidental bridges, and jagged edges.
    Rule 7: Keep the canvas exactly {cw} x {ch} pixels and the artwork filling it exactly as the sketch does (the final artwork is {w} x {h} pixels).
    Rule 8: Final pixels must be PURE RED (255,0,0) or PURE WHITE (255,255,255) only, flat colour, no shading, no texture, no outlines in other colours."""


    def png_bytes(img, name):
        buf = io.BytesIO()
        img.save(buf, "PNG")
        buf.seek(0)
        buf.name = name
        return buf


    def sunburst(img, prompt, cw, ch):
        for extra in ({"size": f"{cw}x{ch}"}, {}):                     # the requested size first, else the model's own
            try:
                t0 = time.time()
                r = client.images.edit(model=IMAGE_MODEL, image=png_bytes(img, "step2.png"), prompt=prompt,
                                       quality=IMAGE_QUALITY, n=1, **extra)
                out = Image.open(io.BytesIO(base64.b64decode(r.data[0].b64_json))).convert("RGB")
                return out, {"params": extra, "seconds": round(time.time() - t0, 1)}
            except BadRequestError as e:
                print("   Sunburst rejected", extra or "defaults", ":", str(e)[:160])
        raise RuntimeError("Sunburst rejected every parameter combination")


    print("=" * 90)
    print("CELL 14: SUNBURST BATCH  -", IMAGE_MODEL, "quality", IMAGE_QUALITY)
    print("=" * 90)
    missing = [d for d in DESIGNS if d not in S2]
    if missing:
        print("No Step 2 sketch in Stage_2 for:", missing)
    for d in DESIGNS:
        if d not in S2:
            continue
        out_png = RAW / f"{d} GNR 3 sunburst{VERSION}.png"
        sketch = Image.open(S2[d]).convert("RGB")
        w, h = sketch.size
        cw, ch, s = canvas_for(w, h)
        prompt = prompt_for(w, h, cw, ch, s)
        if out_png.exists() and not FORCE_REGENERATE:
            print(f"{d}: reusing {out_png.name} (no Sunburst call, no cost)")
            continue
        canvas = sketch.resize((cw, ch), Image.LANCZOS)
        print(f"{d}: Step 2 {w} x {h} -> canvas {cw} x {ch} ... ", end="", flush=True)
        img, info = sunburst(canvas, prompt, cw, ch)
        img.save(out_png)
        (RAW / f"{d} prompt{VERSION}.txt").write_text(prompt, encoding="utf-8")
        a = np.asarray(img).astype(int)
        red = float((1 - (a[..., 1] + a[..., 2]) / 510 >= 0.5).mean())
        meta = {"design": d, "step2": S2[d].name, "step2_size": [w, h], "canvas": [cw, ch], "returned": list(img.size),
                "model": IMAGE_MODEL, "quality": IMAGE_QUALITY, **info, "red_share": round(red, 3)}
        (RAW / f"{d} canvas{VERSION}.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")
        print(f"done in {info['seconds']:.0f}s, returned {img.size[0]} x {img.size[1]}, red {100 * red:.0f}%")
    print("Saved in:", RAW)
    print(sorted(p.name for p in RAW.glob("*sunburst*.png")))

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_PAIR_MODEL_EXPERIMENTS:
    # ==================================================================================================
    # CELL 15A: PAIR MODEL - DATA  -  the designer's own Step 2 -> Step 3 pairs (Stage_2 / Stage_3, read only), cleaned by
    # the pair audit (09_Pair_Model/pairs_manifest.json: kept pairs, repeat shift, split by design: train 72% + val 8% = 80%,
    # test 20%). Loads every kept pair once (cached in /content for re-runs) and defines training crops, model, loss, scoring.
    # Augmentation: random crops, flips and 90-degree turns (sketch and target together, pixel exact); brightness, contrast,
    # gamma, blur, noise and JPEG on the SKETCH ONLY. No free-angle rotation or resizing: they would break the pixel rules.
    # ==================================================================================================
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "timm"], check=True)
    import io
    import json
    import time
    from pathlib import Path
    from concurrent.futures import ThreadPoolExecutor
    import numpy as np
    import torch
    import torch.nn.functional as F
    from PIL import Image, ImageFilter
    from scipy import ndimage as ndi
    from google.colab import drive
    import segmentation_models_pytorch as smp

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DATASET = Path("/content/drive/MyDrive/GNR Design Data-Set")
    PAIRDIR = DATASET / "GNR_Step3_Photoshop_Pipeline" / "09_Pair_Model"
    MANIFEST = json.loads((PAIRDIR / "pairs_manifest.json").read_text())
    DEVICE = "cuda"
    CROP = 256                     # training crops are CROP x CROP pixels
    BATCH = 32
    EDGE_W = 4.0                   # with the edge loss, pixels at a colour change (outlines, 2-px separations) count 4 times
    CACHE = Path("/content/pairs_cache.pt")


    def load_pair(r):
        g = np.asarray(Image.open(DATASET / "Stage_2" / r["step2"]).convert("L"))
        a = np.asarray(Image.open(DATASET / "Stage_3" / r["step3"]).convert("RGB")).astype(np.float32) / 255.0
        red = (1.0 - (a[..., 1] + a[..., 2]) / 2.0) >= 0.5
        dx, dy = r["shift"]                                          # the designer's file moved to the sketch's phase
        red = np.roll(np.roll(red, dy, 0), dx, 1)
        return g, red.astype(np.uint8)


    if CACHE.exists():
        PAIRS = torch.load(CACHE, weights_only=False)
    else:
        t0 = time.time()
        with ThreadPoolExecutor(16) as ex:
            loaded = list(ex.map(load_pair, MANIFEST["pairs"]))
        PAIRS = {r["design"]: gt for r, gt in zip(MANIFEST["pairs"], loaded)}
        torch.save(PAIRS, CACHE)
        print(f"loaded {len(PAIRS)} pairs from Drive in {time.time() - t0:.0f}s")
    SPLIT = {s: [r["design"] for r in MANIFEST["pairs"] if r["split"] == s] for s in ("train", "val", "test")}


    def augment_sketch(g, rng):
        """Only the sketch changes (never the target): brightness / contrast / gamma, blur, noise, JPEG."""
        f = g.astype(np.float32)
        if rng.random() < 0.8:
            f = (f - 128) * rng.uniform(0.75, 1.25) + 128 + rng.uniform(-30, 30)
        if rng.random() < 0.3:
            f = 255 * (np.clip(f, 0, 255) / 255) ** rng.uniform(0.7, 1.4)
        img = Image.fromarray(np.clip(f, 0, 255).astype(np.uint8))
        if rng.random() < 0.25:
            img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.0)))
        if rng.random() < 0.25:
            buf = io.BytesIO()
            img.save(buf, "JPEG", quality=int(rng.integers(40, 95)))
            buf.seek(0)
            img = Image.open(buf).convert("L")
        f = np.asarray(img).astype(np.float32)
        if rng.random() < 0.25:
            f = f + rng.normal(0, rng.uniform(2, 10), f.shape)
        return np.clip(f, 0, 255).astype(np.uint8)


    class Crops(torch.utils.data.Dataset):
        """Random CROP x CROP pieces of the training pairs (bigger designs are picked more often)."""
        def __init__(self, ids, n):
            self.ids, self.n = ids, n
            areas = np.array([PAIRS[d][0].size for d in ids], float)
            self.p = areas / areas.sum()

        def __len__(self):
            return self.n

        def __getitem__(self, i):
            rng = np.random.default_rng()
            g, t = PAIRS[self.ids[rng.choice(len(self.ids), p=self.p)]]
            m = np.ones_like(t)
            ph, pw = max(0, CROP - g.shape[0]), max(0, CROP - g.shape[1])
            if ph or pw:                                             # small design: pad, and leave the padding out of the loss
                g = np.pad(g, ((0, ph), (0, pw)), constant_values=255)
                t = np.pad(t, ((0, ph), (0, pw)))
                m = np.pad(m, ((0, ph), (0, pw)))
            y = int(rng.integers(0, g.shape[0] - CROP + 1))
            x = int(rng.integers(0, g.shape[1] - CROP + 1))
            g, t, m = (a[y:y + CROP, x:x + CROP] for a in (g, t, m))
            k = int(rng.integers(4))
            g, t, m = (np.rot90(a, k) for a in (g, t, m))
            if rng.random() < 0.5:
                g, t, m = (a[:, ::-1] for a in (g, t, m))
            g = augment_sketch(np.ascontiguousarray(g), rng)
            as_t = lambda a, s: torch.from_numpy(np.ascontiguousarray(a).astype(np.float32) / s)[None]
            return as_t(g, 255.0), as_t(t, 1.0), as_t(m, 1.0)


    def build(cfg):
        arch = {"Unet": smp.Unet, "UnetPlusPlus": smp.UnetPlusPlus}[cfg["arch"]]
        return arch(encoder_name=cfg["encoder"], encoder_weights="imagenet", in_channels=1, classes=1).to(DEVICE)


    def edge_weight(t):
        mx = F.max_pool2d(t, 3, 1, 1)
        mn = -F.max_pool2d(-t, 3, 1, 1)
        return 1 + (EDGE_W - 1) * (mx != mn).float()


    def loss_fn(logit, t, m, cfg):
        w = m * edge_weight(t) if cfg["edge"] else m
        bce = (F.binary_cross_entropy_with_logits(logit, t, reduction="none") * w).sum() / w.sum()
        p = torch.sigmoid(logit) * m
        dice = 1 - (2 * (p * t).sum() + 1) / (p.sum() + (t * m).sum() + 1)
        return bce + dice


    def train(cfg, steps, log_every=1000):
        model = build(cfg)
        lr = cfg.get("lr", 5e-4)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
        dl = torch.utils.data.DataLoader(Crops(SPLIT["train"], steps * BATCH), batch_size=BATCH, num_workers=8,
                                         drop_last=True, pin_memory=True)
        model.train()
        t0, run = time.time(), None
        for i, (x, t, m) in enumerate(dl):
            x, t, m = x.to(DEVICE, non_blocking=True), t.to(DEVICE, non_blocking=True), m.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logit = model(x)
            loss = loss_fn(logit.float(), t, m, cfg)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            run = loss.item() if run is None else 0.98 * run + 0.02 * loss.item()
            if (i + 1) % log_every == 0:
                print(f"   {cfg['name']}: step {i + 1}/{steps}  loss {run:.4f}  {time.time() - t0:.0f}s", flush=True)
        return model


    @torch.no_grad()
    def predict(model, g, tile=1024, pad=64):
        """Probability of red for a whole design (big designs in overlapping tiles)."""
        model.eval()
        H, W = g.shape
        out = np.zeros((H, W), np.float32)
        for y0 in range(0, H, tile):
            for x0 in range(0, W, tile):
                y1, x1 = min(H, y0 + tile), min(W, x0 + tile)
                ya, xa, yb, xb = max(0, y0 - pad), max(0, x0 - pad), min(H, y1 + pad), min(W, x1 + pad)
                sub = g[ya:yb, xa:xb]
                h, w = sub.shape
                hp, wp = -(-h // 32) * 32, -(-w // 32) * 32
                x = np.pad(sub, ((0, hp - h), (0, wp - w)), constant_values=255).astype(np.float32) / 255.0
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    p = torch.sigmoid(model(torch.from_numpy(x)[None, None].to(DEVICE)).float())[0, 0, :h, :w].cpu().numpy()
                out[y0:y1, x0:x1] = p[y0 - ya:y1 - ya, x0 - xa:x1 - xa]
        return out


    def scores(pred, t):
        miss, extra = t & ~pred, pred & ~t
        po, pi = ndi.distance_transform_edt(~pred), ndi.distance_transform_edt(pred)
        return {"agree": float((pred == t).mean()), "visible": float(((miss & (po > 1)) | (extra & (pi > 1))).mean()),
                "red": float(pred.mean())}


    def evaluate(model, ids, thresholds=(0.5,), save_dir=None, threshold_for_save=0.5):
        """Mean over designs: agreement with the designer, visible (2+ px) differences, red share; baseline = the sketch's
        black simply recoloured red (no model)."""
        res = {th: [] for th in thresholds}
        base = []
        per_design = {}
        for d in ids:
            g, t = PAIRS[d]
            t = t.astype(bool)
            p = predict(model, g)
            for th in thresholds:
                res[th].append(scores(p >= th, t))
            b = scores(g < 128, t)
            base.append(b)
            per_design[d] = {"model": res[thresholds[0]][-1], "sketch recoloured": b}
            if save_dir is not None:
                pr = p >= threshold_for_save
                a = np.full(pr.shape + (3,), 255, np.uint8)
                a[pr] = (255, 0, 0)
                Image.fromarray(a).save(save_dir / f"{d} GNR 3 - pair model.bmp")
        mean = lambda rows, k: round(100 * float(np.mean([r[k] for r in rows])), 2)
        out = {f"model @{th}": {k: mean(res[th], k) for k in ("agree", "visible", "red")} for th in thresholds}
        out["sketch recoloured"] = {k: mean(base, k) for k in ("agree", "visible", "red")}
        out["designer red"] = round(100 * float(np.mean([PAIRS[d][1].mean() for d in ids])), 2)
        return out, per_design


    print("=" * 90)
    print("CELL 15A: PAIR MODEL - DATA")
    print("=" * 90)
    print("pairs kept:", len(PAIRS), "| train", len(SPLIT["train"]), "| val", len(SPLIT["val"]), "| test", len(SPLIT["test"]))
    print("GPU:", torch.cuda.get_device_name(0))
    x, t, m = Crops(SPLIT["train"], 4)[0]
    print("one training crop:", tuple(x.shape), "sketch range", round(float(x.min()), 2), "-", round(float(x.max()), 2),
          "| red share", round(float(t.mean()), 2))

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_PAIR_MODEL_EXPERIMENTS:
    # ==================================================================================================
    # CELL 15B: PAIR MODEL - WHICH COMBINATION  -  short training of each combination (pre-trained encoder x architecture x
    # loss), scored on the VALIDATION designs (part of the 80% training share; the 20% test designs are not touched here).
    # The best one is trained long in Cell 15C.
    # ==================================================================================================
    CONFIGS = [
        {"name": "Unet resnet34", "arch": "Unet", "encoder": "resnet34", "edge": False},
        {"name": "Unet resnet34 + edge loss", "arch": "Unet", "encoder": "resnet34", "edge": True},
        {"name": "Unet efficientnet-b4 + edge loss", "arch": "Unet", "encoder": "efficientnet-b4", "edge": True},
        {"name": "Unet++ resnet34 + edge loss", "arch": "UnetPlusPlus", "encoder": "resnet34", "edge": True},
    ]
    SCREEN_STEPS = 3000

    print("=" * 90)
    print("CELL 15B: PAIR MODEL - WHICH COMBINATION  (", SCREEN_STEPS, "steps each, scored on", len(SPLIT["val"]), "validation designs )")
    print("=" * 90)
    SCREEN = {}
    for cfg in CONFIGS:
        t0 = time.time()
        model = train(cfg, SCREEN_STEPS)
        summary, _ = evaluate(model, SPLIT["val"])
        SCREEN[cfg["name"]] = {"config": cfg, "minutes": round((time.time() - t0) / 60, 1), **summary}
        s = summary["model @0.5"]
        print(f"{cfg['name']:34s} agreement {s['agree']:.2f}%  visible diff {s['visible']:.2f}%  red {s['red']:.1f}%"
              f"  ({SCREEN[cfg['name']]['minutes']} min)", flush=True)
        del model
        torch.cuda.empty_cache()
    b = SCREEN[CONFIGS[0]["name"]]["sketch recoloured"]
    print(f"{'sketch recoloured (no model)':34s} agreement {b['agree']:.2f}%  visible diff {b['visible']:.2f}%  red {b['red']:.1f}%")
    print(f"{'designer':34s} red {SCREEN[CONFIGS[0]['name']]['designer red']:.1f}%")
    BEST_CFG = max(CONFIGS, key=lambda c: SCREEN[c["name"]]["model @0.5"]["agree"])
    print("best combination:", BEST_CFG["name"])
    (PAIRDIR / "screening.json").write_text(json.dumps(SCREEN, indent=1), encoding="utf-8")

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_PAIR_MODEL_EXPERIMENTS:
    # ==================================================================================================
    # CELL 15C: PAIR MODEL - FINAL TRAINING + TEST  -  the best combination from Cell 15B, trained long on all training
    # designs; the red/white threshold is chosen on the validation designs; then scored ONCE on the 20% test designs it
    # has never seen. Saves the model, the test predictions (BMP) and the scores in 09_Pair_Model.
    # ==================================================================================================
    FINAL_STEPS = 20000
    THRESHOLDS = (0.4, 0.45, 0.5, 0.55, 0.6)

    print("=" * 90)
    print("CELL 15C: PAIR MODEL - FINAL:", BEST_CFG["name"], "|", FINAL_STEPS, "steps")
    print("=" * 90)
    t0 = time.time()
    FINAL = train(BEST_CFG, FINAL_STEPS, log_every=2000)
    tag = BEST_CFG["name"].replace(" ", "_").replace("+", "plus")
    torch.save(FINAL.state_dict(), PAIRDIR / f"pair_model_{tag}.pt")
    print(f"trained in {(time.time() - t0) / 60:.0f} min, saved pair_model_{tag}.pt")

    val, _ = evaluate(FINAL, SPLIT["val"], thresholds=THRESHOLDS)
    TH = max(THRESHOLDS, key=lambda th: val[f"model @{th}"]["agree"])
    print("threshold chosen on the validation designs:", TH, {th: val[f"model @{th}"]["agree"] for th in THRESHOLDS})

    PRED_DIR = PAIRDIR / "test_predictions"
    PRED_DIR.mkdir(parents=True, exist_ok=True)
    test, per_design = evaluate(FINAL, SPLIT["test"], thresholds=(TH,), save_dir=PRED_DIR, threshold_for_save=TH)
    s, b = test[f"model @{TH}"], test["sketch recoloured"]
    print()
    print(f"TEST ({len(SPLIT['test'])} designs never seen in training), mean per design:")
    print(f"   pair model             agreement {s['agree']:.2f}%   visible diff {s['visible']:.2f}%   red {s['red']:.1f}%")
    print(f"   sketch recoloured      agreement {b['agree']:.2f}%   visible diff {b['visible']:.2f}%   red {b['red']:.1f}%")
    print(f"   designer                                                         red {test['designer red']:.1f}%")
    gain = sorted(per_design.items(), key=lambda kv: kv[1]["model"]["agree"] - kv[1]["sketch recoloured"]["agree"])
    better = sum(1 for _, v in gain if v["model"]["agree"] > v["sketch recoloured"]["agree"])
    print(f"   model better than the recoloured sketch on {better} of {len(gain)} test designs")
    print("   least helped:", [(d, round(100 * (v['model']['agree'] - v['sketch recoloured']['agree']), 1)) for d, v in gain[:5]])
    print("   most helped :", [(d, round(100 * (v['model']['agree'] - v['sketch recoloured']['agree']), 1)) for d, v in gain[-5:]])
    (PAIRDIR / "test_results.json").write_text(json.dumps({"config": BEST_CFG, "steps": FINAL_STEPS, "threshold": TH,
                                                           "validation": val, "test": test, "per_design": per_design},
                                                          indent=1), encoding="utf-8")
    print("saved:", PRED_DIR, "and test_results.json")

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_PAIR_MODEL_EXPERIMENTS:
    # ==================================================================================================
    # CELL 16A: PAIR MODEL - QUALITY BANDS + 8-WAY AVERAGING  -  scores outputs by how much a designer would still redraw
    # (pixels 2+ px from the designer's file): good < 5%, touch-up 5-10%, heavy 10-20%, unusable 20%+.
    # 8-way averaging: the model looks at the sketch in all 8 orientations (4 turns x mirror) and the answers are averaged,
    # which calms the red/white flicker where it is unsure. Tried first on the current model (no retraining).
    # Needs Cells 15A (+ 15C, or the saved model).
    # ==================================================================================================
    BANDS = ((5, "good (< 5% to redraw)"), (10, "touch-up (5-10%)"), (20, "heavy (10-20%)"), (101, "unusable (20%+)"))

    if "FINAL" not in globals():
        FINAL = build({"arch": "Unet", "encoder": "resnet34", "edge": True})
        FINAL.load_state_dict(torch.load(PAIRDIR / "pair_model_Unet_resnet34_plus_edge_loss.pt", map_location=DEVICE))


    def predict8(model, g):
        acc = np.zeros(g.shape, np.float32)
        for k in range(4):
            for flip in (False, True):
                x = np.rot90(g, k)
                x = x[:, ::-1] if flip else x
                p = predict(model, np.ascontiguousarray(x))
                p = p[:, ::-1] if flip else p
                acc += np.rot90(p, -k)
        return acc / 8


    def band_of(visible_pct):
        return next(name for top, name in BANDS if visible_pct < top)


    def evaluate2(model, ids, eight=True, thresholds=(0.5,), save_dir=None, threshold_for_save=0.5):
        """Like evaluate, plus the redraw bands; eight = 8-way averaging."""
        res = {th: [] for th in thresholds}
        for d in ids:
            g, t = PAIRS[d]
            t = t.astype(bool)
            p = predict8(model, g) if eight else predict(model, g)
            for th in thresholds:
                res[th].append(scores(p >= th, t))
            if save_dir is not None:
                pr = p >= threshold_for_save
                a = np.full(pr.shape + (3,), 255, np.uint8)
                a[pr] = (255, 0, 0)
                Image.fromarray(a).save(save_dir / f"{d} GNR 3 - pair model.bmp")
        out = {}
        for th in thresholds:
            rows = res[th]
            bands = {name: 0 for _, name in BANDS}
            for r in rows:
                bands[band_of(100 * r["visible"])] += 1
            out[th] = {"agree": round(100 * float(np.mean([r["agree"] for r in rows])), 2),
                       "visible": round(100 * float(np.mean([r["visible"] for r in rows])), 2),
                       "red": round(100 * float(np.mean([r["red"] for r in rows])), 2), "bands": bands,
                       "per design": {d: r for d, r in zip(ids, rows)}}
        return out


    print("=" * 90)
    print("CELL 16A: QUALITY BANDS + 8-WAY AVERAGING  (current model)")
    print("=" * 90)
    TH0 = 0.55                                                     # the threshold Cell 15C chose
    v1 = evaluate2(FINAL, SPLIT["val"], eight=False, thresholds=(TH0,))[TH0]
    v8 = evaluate2(FINAL, SPLIT["val"], eight=True, thresholds=(0.45, 0.5, 0.55, 0.6))
    TH8 = max(v8, key=lambda th: v8[th]["agree"])
    for name, r in (("validation, as before", v1), (f"validation, 8-way @{TH8}", v8[TH8])):
        print(f"{name:28s} agreement {r['agree']:.2f}%  to redraw {r['visible']:.2f}%  bands {r['bands']}")
    t1 = evaluate2(FINAL, SPLIT["test"], eight=False, thresholds=(TH0,))[TH0]
    t8 = evaluate2(FINAL, SPLIT["test"], eight=True, thresholds=(TH8,))[TH8]
    for name, r in (("TEST, as before", t1), (f"TEST, 8-way @{TH8}", t8)):
        print(f"{name:28s} agreement {r['agree']:.2f}%  to redraw {r['visible']:.2f}%  bands {r['bands']}")
    (PAIRDIR / "tta_results.json").write_text(json.dumps({"threshold": TH8, "val before": v1, "val 8-way": v8[TH8],
                                                          "test before": t1, "test 8-way": t8}, indent=1), encoding="utf-8")

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_PAIR_MODEL_EXPERIMENTS:
    # ==================================================================================================
    # CELL 16B: PAIR MODEL v2  -  against the blobs: bigger training pieces (384 px: more context to tell fine line work
    # from solid fill), longer training, and a loss without the overlap (Dice) term that favours solid blobs (tried against
    # the old loss on the validation designs first). Scored with 8-way averaging and the redraw bands; the 20% test designs
    # are used once, at the end. New outputs go to test_predictions_v2 (the first ones are kept). Needs Cells 15A + 16A.
    # ==================================================================================================
    CROP = 384
    BATCH = 24
    TRY_STEPS = 6000
    FINAL_STEPS_V2 = 30000


    def loss_fn(logit, t, m, cfg):
        w = m * edge_weight(t) if cfg["edge"] else m
        bce = (F.binary_cross_entropy_with_logits(logit, t, reduction="none") * w).sum() / w.sum()
        if not cfg.get("dice", True):
            return bce
        p = torch.sigmoid(logit) * m
        dice = 1 - (2 * (p * t).sum() + 1) / (p.sum() + (t * m).sum() + 1)
        return bce + dice


    TRY = [{"name": "v2 384px, edge loss + Dice", "arch": "Unet", "encoder": "resnet34", "edge": True, "dice": True},
           {"name": "v2 384px, edge loss, no Dice", "arch": "Unet", "encoder": "resnet34", "edge": True, "dice": False}]
    print("=" * 90)
    print("CELL 16B: PAIR MODEL v2  (crops", CROP, "| try", TRY_STEPS, "steps each on validation | final", FINAL_STEPS_V2, "steps)")
    print("=" * 90)
    TRIED = {}
    for cfg in TRY:
        t0 = time.time()
        model = train(cfg, TRY_STEPS, log_every=2000)
        r = evaluate2(model, SPLIT["val"], eight=True, thresholds=(0.5,))[0.5]
        TRIED[cfg["name"]] = {k: r[k] for k in ("agree", "visible", "red", "bands")}
        print(f"{cfg['name']:32s} agreement {r['agree']:.2f}%  to redraw {r['visible']:.2f}%  bands {r['bands']}"
              f"  ({(time.time() - t0) / 60:.0f} min)", flush=True)
        del model
        torch.cuda.empty_cache()
    BEST_V2 = min(TRY, key=lambda c: TRIED[c["name"]]["visible"])            # least to redraw on the validation designs
    print("chosen:", BEST_V2["name"])

    t0 = time.time()
    FINAL_V2 = train(BEST_V2, FINAL_STEPS_V2, log_every=5000)
    torch.save(FINAL_V2.state_dict(), PAIRDIR / "pair_model_v2.pt")
    print(f"trained in {(time.time() - t0) / 60:.0f} min, saved pair_model_v2.pt")
    vv = evaluate2(FINAL_V2, SPLIT["val"], eight=True, thresholds=(0.4, 0.45, 0.5, 0.55, 0.6))
    TH_V2 = min(vv, key=lambda th: vv[th]["visible"])
    print("threshold chosen on validation:", TH_V2, {th: (vv[th]["agree"], vv[th]["visible"]) for th in vv})
    PRED_V2 = PAIRDIR / "test_predictions_v2"
    PRED_V2.mkdir(parents=True, exist_ok=True)
    tv = evaluate2(FINAL_V2, SPLIT["test"], eight=True, thresholds=(TH_V2,), save_dir=PRED_V2, threshold_for_save=TH_V2)[TH_V2]
    print()
    print(f"TEST ({len(SPLIT['test'])} designs), v2 with 8-way averaging @{TH_V2}:")
    print(f"   agreement {tv['agree']:.2f}%  to redraw {tv['visible']:.2f}%  red {tv['red']:.1f}%")
    print("   bands:", tv["bands"])
    (PAIRDIR / "test_results_v2.json").write_text(json.dumps({"tried": TRIED, "config": BEST_V2, "steps": FINAL_STEPS_V2, "crop": CROP,
                                                              "threshold": TH_V2, "validation": {str(k): {kk: v[kk] for kk in ("agree", "visible", "red", "bands")} for k, v in vv.items()},
                                                              "test": tv}, indent=1), encoding="utf-8")
    print("saved:", PRED_V2, "and test_results_v2.json")

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14A: STEP A - HAS NAGARAJU DRAWN THIS DESIGN BEFORE?  (run before Cell 14B; free - no Sunburst / OpenAI calls)
    # Every sketch in 00_Sunburst_Input/<BATCH>/ is compared with his earlier designs (the Stage_2/Stage_3 pairs and every
    # earlier batch he handed back). Drawn before -> his earlier Step 3 fitted to the new sketch is saved as
    # 08_Sunburst_Finished/<BATCH>/<name> - from earlier design <X>.bmp, and Cell 14B skips it. New designs go to Sunburst.
    # The list is in '_step A.json' and 'README - Step A.txt' in the same folder. Never overwrites a file.
    # Code: Notebook/code/step_a.py + archive_lookup.py on Drive.
    # ==================================================================================================
    BATCH = "GM batch"
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DS = "/content/drive/MyDrive/GNR Design Data-Set"
    CODE = DS + "/GNR_Step3_Photoshop_Pipeline/Notebook/code/step_a.py"
    !python -u "{CODE}" "{DS}" --batch "{BATCH}"
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14B: SUNBURST FOR A FOLDER OF NEW SKETCHES  -  like Cell 14, for sketches that are not in Stage_2 (e.g. a GM batch).
    # Each sketch gets the rules prompt that fits its kind (picked from the image, printed for checking):
    #   filled  - black / white sketch: black becomes solid red, outline-only motifs stay red lines
    #   pencil  - light pencil drawing: each outlined motif becomes solid red, its inner pencil lines white cut-outs
    #   shaded  - shaded picture: motifs red on a white background, inner details white cut-outs
    #   colour  - coloured motif: each coloured part red, thin white lines where colours meet
    # Raw images go to 07_Sunburst_Raw/<BATCH>/ ("<name> - sunburst.png" + prompt + canvas json). Sunburst never sees a Step 3.
    # Run Cell 14A (Step A) first: sketches NagaRaju has drawn before get his earlier Step 3 there and are skipped here.
    # ==================================================================================================
    BATCH = "GM batch"
    SKETCH_DIR_NAME = "00_Sunburst_Input/" + BATCH            # inside GNR_Step3_Photoshop_Pipeline
    KIND_OVERRIDE = {}                                         # e.g. {"GM 104": "shaded"} if the automatic pick is wrong
    FORCE_REGENERATE = False
    LONG_SIDE = 1440
    IMAGE_MODEL = "gpt-image-2.5-sunburst"
    IMAGE_QUALITY = "high"

    import base64
    import io
    import json
    import time
    from pathlib import Path
    import numpy as np
    from PIL import Image
    from google.colab import drive, userdata
    from openai import OpenAI, BadRequestError

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    PIPE = Path("/content/drive/MyDrive/GNR Design Data-Set/GNR_Step3_Photoshop_Pipeline")
    SKETCHES = PIPE / SKETCH_DIR_NAME
    RAW = PIPE / "07_Sunburst_Raw" / BATCH
    RAW.mkdir(parents=True, exist_ok=True)
    client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))


    def kind_of(img):
        a = np.asarray(img.convert("RGB")).astype(int)
        g = a.mean(-1)
        sat = a.max(-1) - a.min(-1)
        if float((sat > 60).mean()) > 0.05:
            return "colour"
        if float((g < 128).mean()) >= 0.25:
            return "filled" if len(np.unique(a.reshape(-1, 3), axis=0)) < 5000 else "shaded"
        return "pencil"


    RULE2 = {
        "filled": "Rule 2: The background is WHITE. Every motif is RED: motifs drawn solid black in the sketch are solid red, with their inner details as white cut-outs; motifs drawn only as outlines in the sketch stay line drawings (red lines on white), never filled solid. Avoid accidental holes.",
        "pencil": "Rule 2: The sketch is a pencil line drawing. Every motif outlined in pencil becomes a SOLID RED shape; the pencil lines inside a motif (veins, petal divisions, inner details) become thin WHITE cut-out lines inside the red; the background between the motifs stays WHITE. Ignore paper texture, shading, and any straight grid or seam lines that run across the whole sketch: they are not part of the design.",
        "shaded": "Rule 2: The sketch is a shaded picture. The background is WHITE, even where the picture is dark. Every motif is RED, with its inner details (feathers, petals, patterns) as thin white cut-out lines. Ignore watermarks, logos and icons in the corners.",
        "colour": "Rule 2: The sketch is a coloured motif. The background is WHITE. Every coloured part of the motif becomes solid RED; where two differently coloured parts meet, keep a thin white separation line; small dark details become thin white cut-out lines.",
    }


    def canvas_for(w, h):
        s = LONG_SIDE / max(w, h)
        cw, ch = (max(16, int(round(v * s / 16)) * 16) for v in (w, h))
        return cw, ch, s


    def prompt_for(kind, w, h, cw, ch, s):
        two = max(2, int(round(2 * s)))
        return f"""Act as an expert textile designer and production finisher. Convert the supplied Step 2 sketch into a professional Step 3 red-and-white design, regardless of motif type.

    Analyze all design elements, their relationships, repeated families, curves, fills, and intentional white spaces.

    Apply these rules:
    Rule 1: Preserve the overall composition, every motif, its position, size, orientation and relationships exactly as in the sketch. Do not add, remove or move elements.
    {RULE2[kind]}
    Rule 3: Keep independent components separated by white space. Every leaf, petal, bud and flower is separated from its stem and its neighbours by a continuous thin white line of even width (about {two} pixels on this canvas, 2 pixels at final size), the same width everywhere. Preserve deliberate connections and overlaps.
    Rule 4: Shapes are full and bold with even stroke thickness. Keep thin stems and dots as they are: do not merge leaves into stems or turn dots into other shapes.
    Rule 5: Curves are smooth with gradual tapering and clean pointed or rounded ends. Repeated component families keep the same shape, proportions and filling. All dots are perfectly round.
    Rule 6: Eliminate unintended hooks, blobs, stray pixels, broken strokes, protrusions, accidental bridges, and jagged edges.
    Rule 7: Keep the canvas exactly {cw} x {ch} pixels and the artwork filling it exactly as the sketch does (the final artwork is {w} x {h} pixels).
    Rule 8: Final pixels must be PURE RED (255,0,0) or PURE WHITE (255,255,255) only, flat colour, no shading, no texture, no outlines in other colours."""


    def png_bytes(img, name):
        buf = io.BytesIO()
        img.save(buf, "PNG")
        buf.seek(0)
        buf.name = name
        return buf


    def sunburst(img, prompt, cw, ch):
        for extra in ({"size": f"{cw}x{ch}"}, {}):
            try:
                t0 = time.time()
                r = client.images.edit(model=IMAGE_MODEL, image=png_bytes(img, "step2.png"), prompt=prompt,
                                       quality=IMAGE_QUALITY, n=1, **extra)
                out = Image.open(io.BytesIO(base64.b64decode(r.data[0].b64_json))).convert("RGB")
                return out, {"params": extra, "seconds": round(time.time() - t0, 1)}
            except BadRequestError as e:
                print("   Sunburst rejected", extra or "defaults", ":", str(e)[:160])
        raise RuntimeError("Sunburst rejected every parameter combination")


    print("=" * 90)
    print("CELL 14B: SUNBURST FOR", SKETCHES)
    print("=" * 90)
    files = sorted((p for p in SKETCHES.iterdir() if p.suffix.lower() in (".bmp", ".png", ".jpg", ".jpeg")),
                   key=lambda p: " ".join(p.stem.split()))
    STEP_A = PIPE / "08_Sunburst_Finished" / BATCH / "_step A.json"
    drawn_before = {n: r for n, r in json.loads(STEP_A.read_text()).items() if r["result"] == "drawn before"} if STEP_A.exists() else {}
    if not STEP_A.exists():
        print("Step A has not been run for this batch (Cell 14A) - every sketch goes to Sunburst")
    for p in files:
        name = " ".join(p.stem.split())
        if name in drawn_before:
            print(f"{name}: drawn before - his earlier design {drawn_before[name]['closest earlier design']} is used (Step A), "
                  "no Sunburst call")
            continue
        out_png = RAW / f"{name} - sunburst.png"
        sketch = Image.open(p).convert("RGB")
        w, h = sketch.size
        kind = KIND_OVERRIDE.get(name, kind_of(sketch))
        cw, ch, s = canvas_for(w, h)
        if out_png.exists() and not FORCE_REGENERATE:
            print(f"{name}: reusing {out_png.name} (no Sunburst call, no cost)")
            continue
        prompt = prompt_for(kind, w, h, cw, ch, s)
        print(f"{name}: {kind:6s} {w} x {h} -> canvas {cw} x {ch} ... ", end="", flush=True)
        img, info = sunburst(sketch.resize((cw, ch), Image.LANCZOS), prompt, cw, ch)
        img.save(out_png)
        (RAW / f"{name} - prompt.txt").write_text(prompt, encoding="utf-8")
        a = np.asarray(img).astype(int)
        red = float((1 - (a[..., 1] + a[..., 2]) / 510 >= 0.5).mean())
        meta = {"name": name, "sketch": p.name, "kind": kind, "size": [w, h], "canvas": [cw, ch], "returned": list(img.size),
                "model": IMAGE_MODEL, "quality": IMAGE_QUALITY, **info, "red_share": round(red, 3)}
        (RAW / f"{name} - canvas.json").write_text(json.dumps(meta, indent=1), encoding="utf-8")
        print(f"done in {info['seconds']:.0f}s, red {100 * red:.0f}%")
    print("Saved in:", RAW)

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14E: STEP A-2 - HIS MOTIFS FROM EARLIER DESIGNS  (after the finishing of Cell 14B's Sunburst images; free)
    # For each NEW design of <BATCH> (Step A found no earlier twin) with a finished Sunburst image, motifs that NagaRaju has
    # drawn before in other designs are swapped for his own drawing where the safety checks allow. A NEW file is saved:
    # 08_Sunburst_Finished/<BATCH>/<name> - with motifs from earlier design <X>.bmp (the Sunburst + rules file stays).
    # The list is in '_step A-2.json' and 'README - Step A-2.txt'. Never overwrites a file.
    # Code: Notebook/code/step_a2.py + motif_reuse.py + archive_lookup.py + step_a.py on Drive.
    # ==================================================================================================
    BATCH = "GM batch"
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DS = "/content/drive/MyDrive/GNR Design Data-Set"
    CODE = DS + "/GNR_Step3_Photoshop_Pipeline/Notebook/code/step_a2.py"
    !python -u "{CODE}" "{DS}" --batch "{BATCH}"
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14G: THE CHAIN - STEP 2: SUNBURST DRAWS THE SKETCH  (after chain_run.py prep; ~$0.05 per design)
    # Step 1 (chain_run.py prep) copied NagaRaju's earlier drawing wherever the sketch matches his earlier designs and locked
    # it. Here Sunburst draws the sketch of every design not finished at step 1; after the finishing, chain_run.py finish
    # puts the locked parts back exactly (with his 2-px white line). Designs that already have a raw image are skipped.
    # (Tested: giving Sunburst the draft with a mask leaves the open part blank - it cannot see the sketch under the mask.)
    # Code: Notebook/code/chain_sunburst.py
    # ==================================================================================================
    BATCH = "Tile test"
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    CODE = "/content/drive/MyDrive/GNR Design Data-Set/GNR_Step3_Photoshop_Pipeline/Notebook/code/chain_sunburst.py"
    exec(open(CODE, encoding="utf-8").read())
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14H: SUNBURST FOR A BATCH FOLDER, SEVERAL AT A TIME  (~$0.05 per sketch, standard price)
    # Like Cell 14B (prompt chosen per kind of sketch), with WORKERS Sunburst calls at once. Sketches: 00_Sunburst_Input/<BATCH>/
    # Raw images -> 07_Sunburst_Raw/<BATCH>/ (+ tokens billed). Skips sketches that already have a raw image (no cost twice).
    # Stops cleanly if the OpenAI balance runs out: top up and run it again. Code: Notebook/code/sunburst_folder_parallel.py
    # ==================================================================================================
    BATCH = "NagaRaju picks 01"
    WORKERS = 6
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    CODE = "/content/drive/MyDrive/GNR Design Data-Set/GNR_Step3_Photoshop_Pipeline/Notebook/code/sunburst_folder_parallel.py"
    exec(open(CODE, encoding="utf-8").read())
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 17: AUTO-IMPROVE LOOP  -  compare with the designer's Step 3 -> areas to improve -> a corrector learns his
    # finishing -> each design keeps what is closer -> repeat until close (or no more gain). Then the batch joins
    # 11_Design_Repository and the repository corrector (for NEW sketches) is retrained on every pair collected so far.
    # 'heldout' then writes forecast.csv: what a brand-new sketch can expect (designs corrected by a corrector that never
    # saw them). Results: GNR_Step3_Photoshop_Pipeline/10_Auto_Improve/<BATCH>/ (best/, round NN/, scoreboard.csv).
    # New sketches without designer files: MODES = ["apply"] -> 08_Sunburst_Finished/<BATCH>/repository corrected/.
    # Code: Notebook/code/auto_improve.py (the same file runs on the PC). Re-running continues where it stopped.
    # ==================================================================================================
    BATCH = "GM batch"
    NAME_MAP = "GM 108=GM 107A;GM 109=GM 108A"      # designer files named differently from the sketches
    MODES = ["fit", "heldout"]

    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DS = "/content/drive/MyDrive/GNR Design Data-Set"
    PIPE = DS + "/GNR_Step3_Photoshop_Pipeline"
    CODE = PIPE + "/Notebook/code/auto_improve.py"
    WORK = PIPE + "/10_Auto_Improve/_work/" + BATCH
    for MODE in MODES:
        !python -u "{CODE}" "{DS}" "{WORK}" --mode {MODE} --batch "{BATCH}" --refmap "{NAME_MAP}"

else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14D: PENCIL PROMPT TEST  -  20 Sunburst calls (10 GM pencil designs x prompts A and B), about $2.
    # The code is in Notebook/code/cell14d_pencil_test.py on Drive (read the top of that file for what A and B are).
    # Skips images that already exist, so running it again costs nothing.
    # ==================================================================================================
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    CODE = "/content/drive/MyDrive/GNR Design Data-Set/GNR_Step3_Photoshop_Pipeline/Notebook/code/cell14d_pencil_test.py"
    exec(open(CODE, encoding="utf-8").read())
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 14C: SUNBURST THROUGH THE BATCH API  -  Stage_2 sketches drawn by Sunburst at half price (200 pairs ~ $5).
    # The code is in Notebook/code/cell14c_sunburst_batch_api.py on Drive (settings at its top: STAGE, N_NEW, ...).
    # First run: picks the pairs and submits them. Later runs: show progress and save the finished images. Never submits twice.
    # ==================================================================================================
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    CODE = "/content/drive/MyDrive/GNR Design Data-Set/GNR_Step3_Photoshop_Pipeline/Notebook/code/cell14c_sunburst_batch_api.py"
    exec(open(CODE, encoding="utf-8").read())
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 17B: GROW THE REPOSITORY WITH STAGE_2 PAIRS + THE HONEST TEST ON THE GM BATCH
    #   add      - the finished Stage_2 pairs of STAGE join 11_Design_Repository (his files = their Stage_3 files)
    #   heldout  - every GM design corrected by a corrector that never saw his file for it, now also trained on the
    #              repository's Stage_2 pairs -> 10_Auto_Improve/GM batch/forecast + N repository pairs.csv
    #   repository - the corrector for new sketches, retrained on everything in the repository
    # ==================================================================================================
    STAGE = "Stage_2 batch 01"
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DS = "/content/drive/MyDrive/GNR Design Data-Set"
    PIPE = DS + "/GNR_Step3_Photoshop_Pipeline"
    CODE = PIPE + "/Notebook/code/auto_improve.py"
    WORK = PIPE + "/10_Auto_Improve/_work/GM batch"
    !python -u "{CODE}" "{DS}" "{WORK}" --mode add --batch "{STAGE}"
    !python -u "{CODE}" "{DS}" "{WORK}" --mode heldout --batch "GM batch" --refmap "GM 108=GM 107A;GM 109=GM 108A"
    !python -u "{CODE}" "{DS}" "{WORK}" --mode repository
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ==================================================================================================
    # CELL 17C: DOES THE CORRECTOR HELP ON STAGE_2 DESIGNS IT NEVER SAW?  (decides whether more Stage_2 pairs are worth buying)
    #   heldout on STAGE (4 folds; trains on the other Stage_2 folds + the GM batch) -> 10_Auto_Improve/<STAGE>/forecast...csv
    #   --forecast-only: this test does not change the guard for new sketches
    #   then the GM held-out test again from its saved predictions (seconds): guard with the 3-designs-per-kind rule
    # ==================================================================================================
    STAGE = "Stage_2 batch 01"
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DS = "/content/drive/MyDrive/GNR Design Data-Set"
    PIPE = DS + "/GNR_Step3_Photoshop_Pipeline"
    CODE = PIPE + "/Notebook/code/auto_improve.py"
    !python -u "{CODE}" "{DS}" "{PIPE}/10_Auto_Improve/_work/{STAGE}" --mode heldout --batch "{STAGE}" --forecast-only
    !python -u "{CODE}" "{DS}" "{PIPE}/10_Auto_Improve/_work/GM batch" --mode heldout --batch "GM batch" --refmap "GM 108=GM 107A;GM 109=GM 108A"
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ======================================================================================================================
    # CELL 17D: THE BIGGER CORRECTOR LEARNS FROM NAGARAJU'S PICKS  (360 designs: his Stage_2 ranges + the 57 uploads)
    #   add            - the finished designs of BATCH join 11_Design_Repository (his files: Step_3 from Nagaraju/<name>A.bmp)
    #   GM heldout     - the honest test: each GM design corrected by a corrector that never saw his file for it, now trained
    #                    on the repository incl. BATCH -> 10_Auto_Improve/GM batch/forecast ... .csv (this one sets the guard
    #                    that decides which sketch kinds get the corrector)
    #   BATCH heldout  - the same honest test on the 360 picks themselves (forecast only: the guard is not changed)
    #   repository     - the corrector for new sketches, retrained on everything in the repository
    # LEVEL 1 = the bigger corrector (as for the 16 GM images). Needs a GPU runtime; about 1.5-2 hours. If Colab disconnects,
    # run the cell again: finished folds are kept on Drive and are not trained again.
    # ======================================================================================================================
    BATCH = "NagaRaju picks 01"
    LEVEL = 1
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DS = "/content/drive/MyDrive/GNR Design Data-Set"
    PIPE = DS + "/GNR_Step3_Photoshop_Pipeline"
    CODE = PIPE + "/Notebook/code/auto_improve.py"
    GMW = PIPE + "/10_Auto_Improve/_work/GM batch"
    BW = PIPE + "/10_Auto_Improve/_work/" + BATCH
    !python -u "{CODE}" "{DS}" "{GMW}" --mode add --batch "{BATCH}"
    !python -u "{CODE}" "{DS}" "{GMW}" --mode heldout --batch "GM batch" --refmap "GM 108=GM 107A;GM 109=GM 108A" --level {LEVEL}
    !python -u "{CODE}" "{DS}" "{BW}" --mode heldout --batch "{BATCH}" --level {LEVEL} --forecast-only
    !python -u "{CODE}" "{DS}" "{GMW}" --mode repository --level {LEVEL}
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')


In [ ]:
if V5_RUN_EXTERNAL_CELLS:
    # ======================================================================================================================
    # CELL 14I: SUNBURST WITH HIS EXAMPLES  -  every sketch goes to Sunburst together with 3 of his most similar earlier
    #   designs as example pictures (his Step 2 sketch on the left | his finished Step 3 on the right), so Sunburst copies
    #   HOW he converts a sketch like this. About $0.10 per sketch. Test batch 'Examples test 01': 20 sketches, about $2.
    #   Sketches + examples: 00_Sunburst_Input/<BATCH>/ (and _examples/); results: 07_Sunburst_Raw/<BATCH>/.
    #   Already-made images are skipped, so running it again only does what is missing.
    # ======================================================================================================================
    BATCH = "Examples test 01"
    WORKERS = 5
    from pathlib import Path
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    exec(Path("/content/drive/MyDrive/GNR Design Data-Set/GNR_Step3_Photoshop_Pipeline/Notebook/code/sunburst_examples.py").read_text())
else:
    print('Optional experiment skipped. External Drive scripts have not been upgraded or certified by this notebook.')
